## Projektkonfiguration und Datenschemata


In [1]:
CONFIG = {
    "notebook_version": "final-paper-run-code-llm-v2-ablation-patch",
    "code_llm_scaffold_version": "Code-LLM-v2",
    "datasets": ["adult", "pima"],
    "train_sizes": [100, 500],
    "seeds": [42, 123, 2025],
    "methods": ["ctgan", "ctgan_strong", "direct_icl", "code_llm", "fallback_only"],
    "synthetic_rows": 500,
    "smoke_synthetic_rows": 50,
    "ollama_endpoint": "http://localhost:11434",
    "ollama_model": "llama3.2:3b",
    "ollama_top_p": 0.9,
    "code_llm_temperature": 0.1,
    "direct_icl_temperature": 0.5,
    "code_llm_num_predict": 1800,
    "direct_icl_num_predict": 2200,
    "direct_icl_batch_size": 25,
    "max_code_llm_attempts": 5,
    "max_direct_icl_batches": 40,
    "max_refill_attempts": 20,
    "ctgan_epochs": 50,
    "ctgan_strong_epochs": 300,
    "ctgan_raw_multiplier": 5,
    "ctgan_max_raw_multiplier": 20,
    "method_success_min_llm_fraction": 0.5,
    # Audit flags only control additional logging/snapshots; experiment logic and parameters remain unchanged.
    "audit_artifacts_enabled": True,
    "save_raw_llm_outputs": True,
    "save_code_llm_traces": True,
    "save_final_synthetic_csvs": True,
    "save_row_provenance": True,
    "save_per_run_results": True,
    "save_incremental_results": True,
    "auto_create_missing_splits": True,
    "ollama_timeout_seconds": None,
}
ADULT_COLUMNS = ["age", "workclass", "education", "education-num", "marital-status", "occupation", "race", "sex", "hours-per-week", "income"]
PIMA_COLUMNS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
EDUCATION_TO_NUM = {
    "Preschool": 1, "1st-4th": 2, "5th-6th": 3, "7th-8th": 4, "9th": 5, "10th": 6,
    "11th": 7, "12th": 8, "HS-grad": 9, "Some-college": 10, "Assoc-voc": 11,
    "Assoc-acdm": 12, "Bachelors": 13, "Masters": 14, "Prof-school": 15, "Doctorate": 16,
}
ADULT_ALLOWED_VALUES = {
    "workclass": ["Federal-gov", "Local-gov", "Private", "Self-emp-inc", "Self-emp-not-inc", "State-gov", "Without-pay"],
    "education": ["10th", "11th", "12th", "1st-4th", "5th-6th", "7th-8th", "9th", "Assoc-acdm", "Assoc-voc", "Bachelors", "Doctorate", "HS-grad", "Masters", "Preschool", "Prof-school", "Some-college"],
    "marital-status": ["Divorced", "Married-AF-spouse", "Married-civ-spouse", "Married-spouse-absent", "Never-married", "Separated", "Widowed"],
    "occupation": ["Adm-clerical", "Armed-Forces", "Craft-repair", "Exec-managerial", "Farming-fishing", "Handlers-cleaners", "Machine-op-inspct", "Other-service", "Priv-house-serv", "Prof-specialty", "Protective-serv", "Sales", "Tech-support", "Transport-moving"],
    "race": ["Amer-Indian-Eskimo", "Asian-Pac-Islander", "Black", "Other", "White"],
    "sex": ["Female", "Male"],
    "income": ["<=50K", ">50K"],
}
SCHEMA = {
    "adult": {
        "columns": ADULT_COLUMNS,
        "target": "income",
        "positive_label": ">50K",
        "numeric": ["age", "education-num", "hours-per-week"],
        "integer": ["age", "education-num", "hours-per-week"],
        "categorical": ["workclass", "education", "marital-status", "occupation", "race", "sex", "income"],
        "allowed_values": ADULT_ALLOWED_VALUES,
        "ranges": {"age": (17, 90), "education-num": (1, 16), "hours-per-week": (1, 99)},
    },
    "pima": {
        "columns": PIMA_COLUMNS,
        "target": "Outcome",
        "positive_label": 1,
        "numeric": PIMA_COLUMNS,
        "integer": ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "Age", "Outcome"],
        "categorical": [],
        "allowed_values": {"Outcome": [0, 1]},
        "ranges": {
            "Pregnancies": (0, 17), "Glucose": (0, 199), "BloodPressure": (0, 122),
            "SkinThickness": (0, 99), "Insulin": (0, 846), "BMI": (0.0, 67.1),
            "DiabetesPedigreeFunction": (0.078, 2.42), "Age": (21, 81), "Outcome": (0, 1),
        },
    },
}


## Pfade, Setup und gemeinsame Hilfsfunktionen


In [2]:
import ast
import csv
import hashlib
import io
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
import platform
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
try:
    from sklearn.preprocessing import OneHotEncoder
except Exception:
    OneHotEncoder = None
try:
    from sdv.metadata import SingleTableMetadata
    from sdv.single_table import CTGANSynthesizer
except Exception:
    SingleTableMetadata = None
    CTGANSynthesizer = None
def resolve_project_root():
    env_root = os.environ.get("GRUNDPROJEKT_ROOT")
    cwd = Path.cwd().resolve()
    candidates = []
    if env_root:
        candidates.append((Path(env_root).expanduser(), True))
    candidates.extend([(cwd, True), (cwd / "Grundprojekt_Code", True)])
    candidates.extend((parent, False) for parent in cwd.parents)
    for candidate, broad_match in candidates:
        candidate = candidate.expanduser().resolve()
        if not candidate.exists():
            continue
        if candidate.name == "Grundprojekt_Code":
            return candidate
        if broad_match and ((candidate / "data" / "processed").exists() or (candidate / "data" / "splits").exists() or (candidate / "README.md").exists()):
            return candidate
    return cwd

PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SPLIT_DIR = DATA_DIR / "splits"
GENERATED_CODE_DIR = DATA_DIR / "generated_code"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLE_DIR = RESULTS_DIR / "tables"
PLOT_DIR = RESULTS_DIR / "plots"
RUNS_DIR = RESULTS_DIR / "runs"
for directory in [PROCESSED_DIR, SPLIT_DIR, GENERATED_CODE_DIR, TABLE_DIR, PLOT_DIR, RUNS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
def set_all_seeds(seed):
    random.seed(int(seed))
    np.random.seed(int(seed))
    return np.random.default_rng(int(seed))
def planned_experiment_count():
    return len(CONFIG["datasets"]) * len(CONFIG["train_sizes"]) * len(CONFIG["seeds"]) * len(CONFIG["methods"])
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def make_run_id():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def project_relative_path(path):
    if path is None or str(path) == "":
        return ""
    p = Path(path)
    try:
        return p.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except Exception:
        return p.as_posix() if not p.is_absolute() else str(p)

def json_safe(value):
    if isinstance(value, Path):
        return project_relative_path(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if isinstance(value, (pd.Timestamp,)):
        return value.isoformat()
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if pd.isna(value) if isinstance(value, (float, np.floating)) else False:
        return None
    return value

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, ensure_ascii=False), encoding="utf-8")
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def file_hash_record(path):
    path = Path(path)
    return {"path": project_relative_path(path), "bytes": int(path.stat().st_size), "sha256": sha256_file(path)}

def run_artifact_stem(dataset, train_size, seed, method):
    return f"{dataset}_{int(train_size)}_seed{int(seed)}_{method}"

def ensure_audit_run_dirs(run_id=None):
    run_id = run_id or make_run_id()
    run_dir = RUNS_DIR / run_id
    subdirs = {
        "tables": run_dir / "tables",
        "plots": run_dir / "plots",
        "synthetic_final": run_dir / "synthetic_final",
        "raw_llm_direct_icl": run_dir / "raw_llm" / "direct_icl",
        "code_llm_traces": run_dir / "code_llm_traces",
        "row_provenance": run_dir / "row_provenance",
        "run_results": run_dir / "run_results",
    }
    for path in [run_dir, *subdirs.values()]:
        path.mkdir(parents=True, exist_ok=True)
    return {"run_id": run_id, "run_dir": run_dir, **subdirs}

def save_pip_freeze(run_context):
    path = run_context["run_dir"] / "pip_freeze.txt"
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=60)
        body = proc.stdout if proc.returncode == 0 else (proc.stdout + "\n" + proc.stderr)
    except Exception as exc:
        body = f"pip freeze failed: {type(exc).__name__}: {exc}\n"
    path.write_text(body, encoding="utf-8")
    return path

def get_ollama_version(timeout_seconds=5):
    try:
        proc = subprocess.run(["ollama", "--version"], capture_output=True, text=True, timeout=timeout_seconds)
        text = (proc.stdout or proc.stderr or "").strip()
        return text if text else "not available"
    except Exception as exc:
        return f"not available: {type(exc).__name__}"

def get_ollama_model_metadata(timeout_seconds=5):
    model_name = CONFIG.get("ollama_model")
    endpoint = str(CONFIG.get("ollama_endpoint", "")).rstrip("/")
    if not endpoint or not model_name:
        return {"status": "not available"}
    try:
        response = requests.get(endpoint + "/api/tags", timeout=timeout_seconds)
        status_code = response.status_code
        response.raise_for_status()
        models = response.json().get("models", [])
        for model in models:
            if model.get("name") == model_name or model.get("model") == model_name:
                return {
                    "status": "available",
                    "http_status": int(status_code),
                    "name": model.get("name") or model.get("model"),
                    "digest": model.get("digest", "not available"),
                    "size": model.get("size", "not available"),
                    "modified_at": model.get("modified_at", "not available"),
                    "details": model.get("details", {}),
                }
        return {"status": "not found", "http_status": int(status_code), "requested_model": model_name}
    except Exception as exc:
        return {"status": "not available", "error": f"{type(exc).__name__}: {exc}"}


def save_environment_snapshot(run_context):
    pip_freeze_path = save_pip_freeze(run_context)
    cwd = Path.cwd()
    try:
        cwd_project_relative = cwd.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix() or "."
    except Exception:
        cwd_project_relative = "not_under_project_root"
    payload = {
        "created_at": utc_now_iso(),
        "notebook_version": CONFIG.get("notebook_version"),
        "python_version": sys.version,
        "python_executable_basename": Path(sys.executable).name,
        "python_executable_absolute_path_redacted": True,
        "platform": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "cwd_project_relative": cwd_project_relative,
        "project_root_name": PROJECT_ROOT.name,
        "project_root_absolute_path_redacted": True,
        "ollama_endpoint": CONFIG.get("ollama_endpoint"),
        "ollama_model": CONFIG.get("ollama_model"),
        "ollama_version": get_ollama_version(),
        "ollama_model_metadata": get_ollama_model_metadata(),
        "pip_freeze": project_relative_path(pip_freeze_path),
        "path_note": "Absolute local filesystem paths are redacted from this portable environment snapshot.",
    }
    return write_json(run_context["run_dir"] / "environment_snapshot.json", payload)
def save_config_snapshot(run_context):
    payload = {
        "created_at": utc_now_iso(),
        "config": CONFIG,
        "schema": SCHEMA,
        "education_to_num": EDUCATION_TO_NUM,
        "audit_note": "Snapshot only; it does not change experiment parameters, seeds, methods, datasets, or metrics.",
    }
    return write_json(run_context["run_dir"] / "config_snapshot.json", payload)

def save_data_hashes(run_context):
    files = []
    for base in [PROCESSED_DIR, SPLIT_DIR]:
        if base.exists():
            files.extend(sorted(p for p in base.rglob("*") if p.is_file()))
    payload = {"created_at": utc_now_iso(), "files": [file_hash_record(p) for p in files]}
    return write_json(run_context["run_dir"] / "data_hashes.json", payload)

def manifest_skip_path(path, run_context=None):
    rel = project_relative_path(path)
    parts = set(Path(rel).parts)
    if ".venv" in parts or "__pycache__" in parts or ".ipynb_checkpoints" in parts:
        return True
    if "__MACOSX" in parts or Path(rel).name in {".DS_Store", "artifact_manifest.json"}:
        return True
    if any(part.startswith("._") for part in Path(rel).parts):
        return True
    return False

def refresh_artifact_manifest(run_context):
    candidates = []
    for base in [PROJECT_ROOT / "README.md", PROJECT_ROOT / "requirements.txt", PROJECT_ROOT / "llm_tabular_synthesis_abgabe_final.ipynb"]:
        if base.exists():
            candidates.append(base)
    for directory in [PROCESSED_DIR, SPLIT_DIR, GENERATED_CODE_DIR, TABLE_DIR, PLOT_DIR, run_context["run_dir"]]:
        if directory.exists():
            candidates.extend(sorted(p for p in directory.rglob("*") if p.is_file()))
    records = [file_hash_record(p) for p in candidates if not manifest_skip_path(p, run_context)]
    payload = {"created_at": utc_now_iso(), "run_id": run_context["run_id"], "files": records}
    return write_json(run_context["run_dir"] / "artifact_manifest.json", payload)

def create_audit_run_context(run_id=None):
    run_context = ensure_audit_run_dirs(run_id)
    save_environment_snapshot(run_context)
    save_config_snapshot(run_context)
    save_data_hashes(run_context)
    refresh_artifact_manifest(run_context)
    return run_context

def save_direct_icl_batch_log(audit_context, dataset, train_size, seed, batch_index, prompt, response, diagnostics):
    if audit_context is None or not CONFIG.get("save_raw_llm_outputs", True):
        return None
    stem = run_artifact_stem(dataset, train_size, seed, "direct_icl")
    payload = {
        "timestamp": utc_now_iso(),
        "dataset": dataset,
        "train_size": int(train_size),
        "seed": int(seed),
        "method": "direct_icl",
        "batch_index": int(batch_index),
        "model": CONFIG["ollama_model"],
        "temperature": CONFIG["direct_icl_temperature"],
        "top_p": CONFIG["ollama_top_p"],
        "num_predict": CONFIG["direct_icl_num_predict"],
        "prompt": prompt,
        "raw_response": response,
        "parse_diagnostics": diagnostics,
        "raw_rows": int(diagnostics.get("raw_rows", 0)),
        "parsed_rows": int(diagnostics.get("parsed_rows", 0)),
        "valid_rows": int(diagnostics.get("valid_rows", 0)),
        "invalid_lines_limited": diagnostics.get("invalid_lines", [])[:10],
        "invalid_reason_counts": diagnostics.get("invalid_reason_counts", {}),
        "ollama_call_metadata": diagnostics.get("ollama_call_metadata", {}),
        "http_status": diagnostics.get("ollama_call_metadata", {}).get("http_status"),
        "request_elapsed_seconds": diagnostics.get("ollama_call_metadata", {}).get("elapsed_seconds"),
    }
    return write_json(audit_context["raw_llm_direct_icl"] / f"{stem}_batch{int(batch_index):03d}.json", payload)

def save_code_llm_attempt_trace(audit_context, dataset, train_size, seed, attempt_index, trace):
    if audit_context is None or not CONFIG.get("save_code_llm_traces", True):
        return None
    stem = run_artifact_stem(dataset, train_size, seed, "code_llm")
    payload = {"timestamp": utc_now_iso(), "dataset": dataset, "train_size": int(train_size), "seed": int(seed), "method": "code_llm", "attempt_index": int(attempt_index), **trace}
    return write_json(audit_context["code_llm_traces"] / f"{stem}_attempt{int(attempt_index):02d}.json", payload)

def save_code_llm_fallback_completion_trace(audit_context, dataset, train_size, seed, final_rows, fallback_rows, previous_failure_stage, previous_failure_reason):
    if audit_context is None or not CONFIG.get("save_code_llm_traces", True):
        return None
    stem = run_artifact_stem(dataset, train_size, seed, "code_llm")
    payload = {
        "timestamp": utc_now_iso(),
        "dataset": dataset,
        "train_size": int(train_size),
        "seed": int(seed),
        "method": "code_llm",
        "failure_stage": "fallback_completion",
        "previous_failure_stage": previous_failure_stage,
        "previous_failure_reason": previous_failure_reason,
        "final_rows": int(final_rows),
        "fallback_rows": int(fallback_rows),
    }
    return write_json(audit_context["code_llm_traces"] / f"{stem}_fallback_completion.json", payload)

def save_final_synthetic_csv(audit_context, synthetic_df, dataset, train_size, seed, method):
    if audit_context is None or not CONFIG.get("save_final_synthetic_csvs", True):
        return None
    stem = run_artifact_stem(dataset, train_size, seed, method)
    output = normalize_to_schema(synthetic_df, dataset).loc[:, schema_for(dataset)["columns"]]
    path = audit_context["synthetic_final"] / f"{stem}.csv"
    output.to_csv(path, index=False)
    return path

def save_row_provenance_sidecar(audit_context, result, dataset, train_size, seed, method, final_rows):
    if audit_context is None or not CONFIG.get("save_row_provenance", True):
        return None
    fallback_rows = int(result.get("fallback_rows", result.get("fallback_or_refill_rows", 0)) or 0)
    primary_rows = max(0, int(final_rows) - fallback_rows)
    primary_source = result.get("provenance_primary_source") or result.get("effective_generator") or method
    fallback_source = result.get("provenance_fallback_source") or result.get("repair_source") or "refill_fallback"
    stem = run_artifact_stem(dataset, train_size, seed, method)
    row_records = []
    for row_id in range(int(final_rows)):
        is_refill = row_id >= primary_rows
        row_records.append({
            "row_id": int(row_id),
            "dataset": dataset,
            "train_size": int(train_size),
            "seed": int(seed),
            "method": method,
            "source": fallback_source if is_refill else primary_source,
            "primary_method": method,
            "is_refill": bool(is_refill),
            "refill_attempt": int(result.get("refill_attempts", 0) or 0) if is_refill else 0,
        })
    row_csv_path = audit_context["row_provenance"] / f"{stem}_row_provenance.csv"
    pd.DataFrame(row_records).to_csv(row_csv_path, index=False)
    payload = {
        "dataset": dataset,
        "train_size": int(train_size),
        "seed": int(seed),
        "method": method,
        "final_rows": int(final_rows),
        "primary_generator_rows": int(primary_rows),
        "fallback_or_refill_rows": int(fallback_rows),
        "fallback_fraction": float(result.get("fallback_fraction", fallback_rows / final_rows if final_rows else 0.0) or 0.0),
        "method_success": bool(result.get("method_success", False)),
        "method_success_threshold": float(CONFIG["method_success_min_llm_fraction"]),
        "row_level_provenance_csv": project_relative_path(row_csv_path),
        "primary_source": primary_source,
        "fallback_source": fallback_source,
        "provenance_note": "fallback_fraction is output provenance, not a quality metric.",
    }
    return write_json(audit_context["row_provenance"] / f"{stem}_provenance.json", payload)

def save_run_result_json(audit_context, result, dataset, train_size, seed, method, extra=None):
    if audit_context is None or not CONFIG.get("save_per_run_results", True):
        return None
    payload = dict(result)
    if extra:
        payload["audit_artifacts"] = extra
    stem = run_artifact_stem(dataset, train_size, seed, method)
    return write_json(audit_context["run_results"] / f"{stem}_result.json", payload)

def append_incremental_result_csv(audit_context, result):
    if audit_context is None:
        return None
    path = audit_context["run_results"] / "incremental_results.csv"
    new = pd.DataFrame([json_safe(result)])
    if path.exists():
        previous = pd.read_csv(path)
        new = pd.concat([previous, new], ignore_index=True, sort=False)
    new.to_csv(path, index=False)
    return path

def snapshot_result_tables_and_plots(run_context):
    if run_context is None:
        return []
    copied = []
    for src in REQUIRED_TABLES if "REQUIRED_TABLES" in globals() else []:
        if Path(src).exists():
            dst = run_context["tables"] / Path(src).name
            shutil.copy2(src, dst)
            copied.append(dst)
    for src in REQUIRED_PLOTS if "REQUIRED_PLOTS" in globals() else []:
        if Path(src).exists():
            dst = run_context["plots"] / Path(src).name
            shutil.copy2(src, dst)
            copied.append(dst)
    refresh_artifact_manifest(run_context)
    return copied

## Eingabedaten und Support-Dateien prüfen


In [3]:
def schema_for(dataset):
    if dataset not in SCHEMA:
        raise ValueError(f"Unknown dataset: {dataset}")
    return SCHEMA[dataset]

def load_processed_dataset(dataset):
    ensure_project_inputs(require_splits=False)
    path = PROCESSED_DIR / f"{dataset}_processed.csv"
    if not path.exists():
        raise FileNotFoundError(project_relative_path(path))
    return pd.read_csv(path)

def load_train_test_split(dataset, train_size, seed):
    ensure_project_inputs(require_splits=True)
    train_path = SPLIT_DIR / f"{dataset}_train_{train_size}_seed{seed}.csv"
    test_path = SPLIT_DIR / f"{dataset}_test.csv"
    if not train_path.exists():
        raise FileNotFoundError(project_relative_path(train_path))
    if not test_path.exists():
        raise FileNotFoundError(project_relative_path(test_path))
    return pd.read_csv(train_path), pd.read_csv(test_path)

def required_processed_paths():
    return [PROCESSED_DIR / f"{dataset}_processed.csv" for dataset in CONFIG["datasets"]]

def required_split_paths():
    paths = []
    for dataset in CONFIG["datasets"]:
        paths.append(SPLIT_DIR / f"{dataset}_test.csv")
        for train_size in CONFIG["train_sizes"]:
            for seed in CONFIG["seeds"]:
                paths.append(SPLIT_DIR / f"{dataset}_train_{train_size}_seed{seed}.csv")
    return paths

def describe_input_search_context():
    zip_candidates = []
    for path in candidate_project_zips():
        zip_candidates.append(project_relative_path(path))
        if len(zip_candidates) >= 8:
            break
    root_candidates = []
    for path in candidate_input_roots():
        root_candidates.append(project_relative_path(path))
        if len(root_candidates) >= 8:
            break
    return {
        "project_root": project_relative_path(PROJECT_ROOT),
        "cwd": project_relative_path(Path.cwd().resolve()),
        "zip_candidates": zip_candidates,
        "directory_candidates": root_candidates,
        "hint": (
            "Lege entweder die Projekt-ZIP-Datei neben das Notebook, entpacke das Projekt so, "
            "dass data/processed/*.csv vorhanden ist, oder setze GRUNDPROJEKT_ROOT bzw. "
            "GRUNDPROJEKT_ZIP auf den Projektordner bzw. die ZIP-Datei."
        ),
    }


def missing_paths_message(paths, label):
    missing = [project_relative_path(path) for path in paths if not Path(path).exists()]
    if not missing:
        return ""
    context = describe_input_search_context()
    lines = [f"Missing {label}:", *missing, "", "Input-Bootstrap-Hinweis:", context["hint"]]
    if context["zip_candidates"]:
        lines.extend(["", "Gefundene ZIP-Kandidaten:", *context["zip_candidates"]])
    else:
        lines.extend(["", "Gefundene ZIP-Kandidaten: keine"])
    if context["directory_candidates"]:
        lines.extend(["", "Geprüfte Projektordner-Kandidaten:", *context["directory_candidates"]])
    return "\n".join(lines)


def check_paths_exist(paths, label):
    message = missing_paths_message(paths, label)
    if message:
        raise FileNotFoundError(message)
    return {"ok": True}

def candidate_input_roots():
    seen = set()
    roots = [PROJECT_ROOT, PROJECT_ROOT.parent, Path.cwd().resolve(), Path.cwd().resolve().parent]
    roots.extend(Path.cwd().resolve().parents)
    if os.environ.get("GRUNDPROJEKT_ROOT"):
        roots.insert(0, Path(os.environ["GRUNDPROJEKT_ROOT"]).expanduser().resolve())
    for root in roots:
        for candidate in [root, root / "Grundprojekt_Code", root.parent / "Grundprojekt_Code"]:
            try:
                candidate = candidate.resolve()
            except Exception:
                continue
            if candidate in seen or candidate == PROJECT_ROOT.resolve():
                continue
            seen.add(candidate)
            yield candidate

def copy_project_inputs_from_directory():
    for candidate in candidate_input_roots():
        source_processed = candidate / "data" / "processed"
        source_splits = candidate / "data" / "splits"
        if not all((source_processed / f"{dataset}_processed.csv").exists() for dataset in CONFIG["datasets"]):
            continue
        for src_path in source_processed.glob("*_processed.csv"):
            dst_path = PROCESSED_DIR / src_path.name
            if not dst_path.exists():
                dst_path.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_path, dst_path)
        if source_splits.exists():
            for src_path in source_splits.glob("*.csv"):
                dst_path = SPLIT_DIR / src_path.name
                if not dst_path.exists():
                    dst_path.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src_path, dst_path)
        copy_project_support_from_directory(candidate)
        return {"ok": True, "source": project_relative_path(candidate)}
    return {"ok": False}

def candidate_project_zips():
    roots = []
    env_zip = os.environ.get("GRUNDPROJEKT_ZIP")
    explicit_paths = []
    if env_zip:
        explicit_paths.append(Path(env_zip).expanduser())
    for root in [PROJECT_ROOT, PROJECT_ROOT.parent, Path.cwd().resolve(), Path.cwd().resolve().parent, *Path.cwd().resolve().parents]:
        try:
            root = root.resolve()
        except Exception:
            continue
        if root not in roots:
            roots.append(root)
    paths = explicit_paths[:]
    patterns = [
        "Abgabe_Grundprojekt_Code_final*.zip",
        "Grundprojekt_Code_final*.zip",
        "Grundprojekt_Code*.zip",
        "*Grundprojekt*Code*.zip",
        "*.zip",
    ]
    for root in roots:
        for pattern in patterns:
            paths.extend(sorted(root.glob(pattern)))
    seen = set()
    for path in paths:
        try:
            path = path.resolve()
        except Exception:
            continue
        if path in seen or not path.is_file():
            continue
        seen.add(path)
        yield path

def zip_member_is_processed_csv(normalized):
    return normalized.endswith(".csv") and ("/data/processed/" in normalized or normalized.startswith("data/processed/"))


def zip_member_is_split_csv(normalized):
    return normalized.endswith(".csv") and ("/data/splits/" in normalized or normalized.startswith("data/splits/"))


def zip_member_is_support_file(normalized):
    return normalized.endswith("README.md") or normalized.endswith("requirements.txt")


def extract_project_inputs_from_zip():
    for zip_path in candidate_project_zips():
        try:
            with zipfile.ZipFile(zip_path) as zf:
                names = zf.namelist()
                normalized_names = [name.replace("\\", "/") for name in names]
                if not all(any(name.endswith(f"data/processed/{dataset}_processed.csv") for name in normalized_names) for dataset in CONFIG["datasets"]):
                    continue
                for name, normalized in zip(names, normalized_names):
                    if zip_member_is_processed_csv(normalized):
                        dst_path = PROCESSED_DIR / Path(normalized).name
                    elif zip_member_is_split_csv(normalized):
                        dst_path = SPLIT_DIR / Path(normalized).name
                    elif zip_member_is_support_file(normalized):
                        dst_path = PROJECT_ROOT / Path(normalized).name
                    else:
                        continue
                    if not dst_path.exists():
                        dst_path.parent.mkdir(parents=True, exist_ok=True)
                        with zf.open(name) as source, open(dst_path, "wb") as target:
                            shutil.copyfileobj(source, target)
            return {"ok": True, "source": project_relative_path(zip_path)}
        except zipfile.BadZipFile:
            continue
    return {"ok": False}


def project_support_paths():
    return [PROJECT_ROOT / "README.md", PROJECT_ROOT / "requirements.txt"]


def copy_project_support_from_directory(candidate):
    copied = []
    for filename in ["README.md", "requirements.txt"]:
        src_path = candidate / filename
        dst_path = PROJECT_ROOT / filename
        if src_path.exists() and not dst_path.exists():
            shutil.copy2(src_path, dst_path)
            copied.append(filename)
    return copied


def copy_project_support_from_directories():
    for candidate in candidate_input_roots():
        copy_project_support_from_directory(candidate)
        if all(path.exists() for path in project_support_paths()):
            return {"ok": True, "source": project_relative_path(candidate)}
    return {"ok": False}


def extract_project_support_from_zip():
    for zip_path in candidate_project_zips():
        try:
            with zipfile.ZipFile(zip_path) as zf:
                for name in zf.namelist():
                    normalized = name.replace("\\", "/")
                    if not zip_member_is_support_file(normalized):
                        continue
                    dst_path = PROJECT_ROOT / Path(normalized).name
                    if not dst_path.exists():
                        with zf.open(name) as source, open(dst_path, "wb") as target:
                            shutil.copyfileobj(source, target)
            if all(path.exists() for path in project_support_paths()):
                return {"ok": True, "source": project_relative_path(zip_path)}
        except zipfile.BadZipFile:
            continue
    return {"ok": False}


def ensure_project_support_files():
    if all(path.exists() for path in project_support_paths()):
        return {"ok": True, "source": "existing"}
    for loader in [copy_project_support_from_directories, extract_project_support_from_zip]:
        result = loader()
        if result.get("ok") and all(path.exists() for path in project_support_paths()):
            return result
    check_paths_exist(project_support_paths(), "project support files")
    return {"ok": True, "source": "existing"}


def ensure_project_inputs(require_splits=True):
    if all(path.exists() for path in required_processed_paths()):
        if not require_splits or all(path.exists() for path in required_split_paths()):
            return {"ok": True, "source": "existing"}
    for loader in [copy_project_inputs_from_directory, extract_project_inputs_from_zip]:
        result = loader()
        if result.get("ok") and all(path.exists() for path in required_processed_paths()):
            if require_splits and not all(path.exists() for path in required_split_paths()):
                create_low_data_splits(rebuild=False)
            if not require_splits or all(path.exists() for path in required_split_paths()):
                return result
    check_paths_exist(required_processed_paths(), "processed input files")
    if require_splits:
        check_paths_exist(required_split_paths(), "split input files")
    return {"ok": True, "source": "existing"}

def check_required_processed_inputs():
    ensure_project_inputs(require_splits=False)
    return check_paths_exist(required_processed_paths(), "processed input files")

def check_required_inputs():
    ensure_project_inputs(require_splits=True)
    return check_paths_exist(required_processed_paths() + required_split_paths(), "processed/split input files")


## Low-Data-Splits erstellen


In [4]:
def create_low_data_splits(rebuild=False):
    for dataset in CONFIG["datasets"]:
        test_path = SPLIT_DIR / f"{dataset}_test.csv"
        train_paths = [
            SPLIT_DIR / f"{dataset}_train_{train_size}_seed{seed}.csv"
            for train_size in CONFIG["train_sizes"]
            for seed in CONFIG["seeds"]
        ]
        split_paths = [test_path] + train_paths
        missing_split_paths = [path for path in split_paths if not path.exists()]
        missing_train_paths = [path for path in train_paths if not path.exists()]
        if not rebuild:
            if test_path.exists() and missing_train_paths:
                raise RuntimeError("Partial split regeneration is disabled. Use the archived splits or rebuild all splits together.")
            if missing_split_paths and len(missing_split_paths) != len(split_paths):
                raise RuntimeError("Partial split regeneration is disabled. Use the archived splits or rebuild all splits together.")
            if not missing_split_paths:
                continue
        data = load_processed_dataset(dataset)
        target = schema_for(dataset)["target"]
        train_pool, test_df = train_test_split(
            data,
            test_size=0.2,
            random_state=0,
            stratify=data[target] if data[target].nunique() > 1 else None,
        )
        test_df.to_csv(test_path, index=False)
        for train_size in CONFIG["train_sizes"]:
            for seed in CONFIG["seeds"]:
                split_path = SPLIT_DIR / f"{dataset}_train_{train_size}_seed{seed}.csv"
                if rebuild or not split_path.exists():
                    stratify = train_pool[target] if train_pool[target].nunique() > 1 else None
                    split, _ = train_test_split(train_pool, train_size=train_size, random_state=seed, stratify=stratify)
                    split.to_csv(split_path, index=False)
    return check_required_inputs()


## Daten normalisieren und synthetische Daten prüfen


In [5]:
def target_to_binary(series, dataset):
    if dataset == "adult":
        return (series.astype(str) == schema_for(dataset)["positive_label"]).astype(int)
    return pd.to_numeric(series, errors="coerce").fillna(0).astype(int).clip(0, 1)
def normalize_to_schema(df, dataset, train_df=None):
    schema = schema_for(dataset)
    out = pd.DataFrame(df).copy()
    for col in schema["columns"]:
        if col not in out.columns:
            out[col] = np.nan
    out = out[schema["columns"]].copy()
    if dataset == "adult":
        for col in schema["categorical"]:
            out[col] = out[col].astype(str).str.strip()
            out.loc[out[col].isin(["", "nan", "NaN", "None"]), col] = np.nan
        out["education-num"] = out["education"].map(EDUCATION_TO_NUM)
        for col in ["age", "hours-per-week"]:
            low, high = schema["ranges"][col]
            out[col] = pd.to_numeric(out[col], errors="coerce").clip(low, high).round().astype("Int64")
        out["education-num"] = pd.to_numeric(out["education-num"], errors="coerce").clip(1, 16).round().astype("Int64")
    if dataset == "pima":
        for col in schema["columns"]:
            low, high = schema["ranges"][col]
            out[col] = pd.to_numeric(out[col], errors="coerce").clip(low, high)
            if col in schema["integer"]:
                out[col] = out[col].round().astype("Int64")
        out["Outcome"] = out["Outcome"].fillna(0).round().astype("Int64").clip(0, 1)
    return out
def row_keys(df, dataset):
    return normalize_to_schema(df, dataset).astype(str).fillna("<NA>").agg("||".join, axis=1)
def validation_masks(df, dataset):
    schema = schema_for(dataset)
    norm = normalize_to_schema(df, dataset)
    complete = norm.notna().all(axis=1)
    category = pd.Series(True, index=norm.index)
    for col, allowed in schema["allowed_values"].items():
        category &= norm[col].isin(allowed)
    ranges = pd.Series(True, index=norm.index)
    for col, (low, high) in schema["ranges"].items():
        values = pd.to_numeric(norm[col], errors="coerce")
        ranges &= values.notna() & values.between(low, high)
    constraint = pd.Series(True, index=norm.index)
    if dataset == "adult":
        constraint &= norm["education-num"].astype("Int64").eq(norm["education"].map(EDUCATION_TO_NUM).astype("Int64"))
    target = norm[schema["target"]].notna()
    if schema["target"] in schema["allowed_values"]:
        target &= norm[schema["target"]].isin(schema["allowed_values"][schema["target"]])
    valid = complete & category & ranges & constraint & target
    return {"normalized": norm, "complete": complete, "category": category, "range": ranges, "constraint": constraint, "target": target, "valid": valid}
def validate_synthetic_data(df, train_df, dataset):
    raw = pd.DataFrame(df)
    masks = validation_masks(raw, dataset)
    norm = masks["normalized"]
    keys = row_keys(norm, dataset)
    train_key_set = set(row_keys(train_df, dataset)) if train_df is not None else set()
    internal_duplicates = keys.duplicated()
    train_duplicates = keys.isin(train_key_set)
    return {
        "format_valid": bool(len(raw) > 0),
        "schema_valid": bool(list(norm.columns) == schema_for(dataset)["columns"]),
        "category_valid": bool(masks["category"].all()),
        "range_valid": bool(masks["range"].all()),
        "constraint_valid": bool(masks["constraint"].all()),
        "target_valid": bool(masks["target"].all()),
        "duplicate_valid": bool((~internal_duplicates).all()),
        "privacy_valid": bool((~train_duplicates).all()),
        "raw_row_count": int(len(raw)),
        "raw_valid_row_count": int(masks["valid"].sum()),
        "final_valid_row_count": int(masks["valid"].sum()),
        "invalid_format_count": int((~masks["complete"]).sum()),
        "invalid_category_count": int((~masks["category"]).sum()),
        "invalid_range_count": int((~masks["range"]).sum()),
        "invalid_constraint_count": int((~masks["constraint"]).sum()),
        "invalid_target_count": int((~masks["target"]).sum()),
        "invalid_duplicate_count": int(internal_duplicates.sum()),
        "unknown_category_rate": float((~masks["category"]).mean()) if len(raw) else np.nan,
        "constraint_violation_rate": float((~masks["constraint"]).mean()) if len(raw) else np.nan,
        "invalid_row_rate": float((~masks["valid"]).mean()) if len(raw) else np.nan,
        "duplicate_rate": float(internal_duplicates.mean()) if len(raw) else np.nan,
        "exact_train_duplicate_rate": float(train_duplicates.mean()) if len(raw) else np.nan,
        "unique_row_fraction": float(keys.nunique() / len(keys)) if len(keys) else np.nan,
    }
def filter_valid_rows(df, train_df, dataset, remove_train_duplicates=True):
    masks = validation_masks(df, dataset)
    valid = masks["normalized"].loc[masks["valid"]].copy()
    before_internal = len(valid)
    valid = valid.drop_duplicates()
    removed_internal = before_internal - len(valid)
    removed_train = 0
    if remove_train_duplicates and train_df is not None and len(valid):
        train_keys = set(row_keys(train_df, dataset))
        keep = ~row_keys(valid, dataset).isin(train_keys)
        removed_train = int((~keep).sum())
        valid = valid.loc[keep].copy()
    return valid.reset_index(drop=True), {"removed_internal_duplicates": int(removed_internal), "removed_train_duplicates": int(removed_train)}

## Statistikbasierte Fallback-Generatoren


In [6]:
def empirical_probs(series, allowed=None, alpha=1.0):
    values = list(allowed) if allowed is not None else sorted(series.dropna().unique().tolist())
    if not values:
        return [], []
    counts = series.value_counts().reindex(values, fill_value=0).astype(float) + alpha
    probs = counts / counts.sum()
    return values, probs.to_numpy(dtype=float)
def numeric_stats(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        s = pd.Series([0.0, 1.0])
    q = s.quantile([0.05, 0.25, 0.50, 0.75, 0.95])
    std = float(s.std(ddof=0))
    return {"mean": float(s.mean()), "std": std if std > 0 else 1.0, "min": float(s.min()), "q05": float(q.loc[0.05]), "q25": float(q.loc[0.25]), "q50": float(q.loc[0.50]), "q75": float(q.loc[0.75]), "q95": float(q.loc[0.95]), "max": float(s.max())}
def get_column_statistics(train_df, dataset):
    schema = schema_for(dataset)
    stats = {"numeric": {}, "categorical": {}, "target_probs": {}}
    for col in schema["numeric"]:
        stats["numeric"][col] = numeric_stats(train_df[col])
    for col in schema["categorical"]:
        values, probs = empirical_probs(train_df[col], schema["allowed_values"].get(col), alpha=1.0)
        stats["categorical"][col] = {"values": values, "probs": probs.tolist()}
    target = schema["target"]
    values, probs = empirical_probs(train_df[target], schema["allowed_values"].get(target), alpha=1.0)
    stats["target_probs"] = {"values": values, "probs": probs.tolist()}
    return stats
def sample_categorical(rng, values, probs=None):
    if isinstance(values, dict):
        probs = values.get("probs", probs)
        values = values.get("values", [])
    if isinstance(values, str) or not hasattr(values, "__iter__"):
        values = [values]
    values = list(values) if values is not None else []
    if not values:
        return np.nan
    try:
        p = np.asarray(probs, dtype=float) if probs is not None else np.ones(len(values), dtype=float)
    except Exception:
        p = np.ones(len(values), dtype=float)
    if p.ndim != 1 or len(p) != len(values) or not np.all(np.isfinite(p)) or np.any(p < 0) or float(p.sum()) <= 0.0:
        p = np.ones(len(values), dtype=float)
    p = p / p.sum()
    return values[int(rng.choice(len(values), p=p))]
def sample_numeric_quantile_jitter(rng, stats, integer=False):
    anchors = np.array([stats["q05"], stats["q25"], stats["q50"], stats["q75"], stats["q95"]], dtype=float)
    value = float(rng.choice(anchors)) + float(rng.normal(0, max((stats["q95"] - stats["q05"]) / 12, 1e-6)))
    value = float(np.clip(value, stats["min"], stats["max"]))
    return int(round(value)) if integer else value
def sample_truncated_normal(rng, mean, std=None, low=None, high=None, integer=False):
    if isinstance(mean, dict):
        stats = mean
        mean = stats.get("mean", stats.get("q50", 0.0))
        std = stats.get("std", std)
        if std is None:
            std = max((float(stats.get("q95", mean)) - float(stats.get("q05", mean))) / 4.0, 1e-6)
        low = stats.get("low", stats.get("min", stats.get("q05", low)))
        high = stats.get("high", stats.get("max", stats.get("q95", high)))
    mean = float(0.0 if mean is None or pd.isna(mean) else mean)
    std = float(1e-6 if std is None or pd.isna(std) else std)
    std = max(std, 1e-6)
    if low is None or pd.isna(low):
        low = mean - 4.0 * std
    if high is None or pd.isna(high):
        high = mean + 4.0 * std
    low, high = float(low), float(high)
    if low > high:
        low, high = high, low
    value = float(np.clip(rng.normal(mean, std), low, high))
    return int(round(value)) if integer else value
def conditional_probs(train_df, condition_col, condition_value, target_col, allowed):
    subset = train_df[train_df[condition_col] == condition_value]
    if len(subset) < 3:
        subset = train_df
    return empirical_probs(subset[target_col], allowed, alpha=1.0)
def make_adult_fallback_rows(train_df, n, seed):
    rng = set_all_seeds(seed)
    stats = get_column_statistics(train_df, "adult")
    rows = []
    for _ in range(n):
        education = sample_categorical(rng, stats["categorical"]["education"]["values"], stats["categorical"]["education"]["probs"])
        workclass = sample_categorical(rng, stats["categorical"]["workclass"]["values"], stats["categorical"]["workclass"]["probs"])
        occ_values, occ_probs = conditional_probs(train_df, "workclass", workclass, "occupation", ADULT_ALLOWED_VALUES["occupation"])
        rows.append({
            "age": sample_numeric_quantile_jitter(rng, stats["numeric"]["age"], True),
            "workclass": workclass,
            "education": education,
            "education-num": EDUCATION_TO_NUM.get(education, 9),
            "marital-status": sample_categorical(rng, stats["categorical"]["marital-status"]["values"], stats["categorical"]["marital-status"]["probs"]),
            "occupation": sample_categorical(rng, occ_values, occ_probs),
            "race": sample_categorical(rng, stats["categorical"]["race"]["values"], stats["categorical"]["race"]["probs"]),
            "sex": sample_categorical(rng, stats["categorical"]["sex"]["values"], stats["categorical"]["sex"]["probs"]),
            "hours-per-week": sample_numeric_quantile_jitter(rng, stats["numeric"]["hours-per-week"], True),
            "income": sample_categorical(rng, stats["target_probs"]["values"], stats["target_probs"]["probs"]),
        })
    return normalize_to_schema(pd.DataFrame(rows), "adult", train_df)
def make_pima_fallback_rows(train_df, n, seed):
    rng = set_all_seeds(seed)
    schema = schema_for("pima")
    stats = get_column_statistics(train_df, "pima")
    rows = []
    for _ in range(n):
        outcome = int(sample_categorical(rng, stats["target_probs"]["values"], stats["target_probs"]["probs"]))
        subset = train_df[train_df["Outcome"] == outcome]
        if len(subset) < 5:
            subset = train_df
        row = {"Outcome": outcome}
        for col in PIMA_COLUMNS:
            if col == "Outcome":
                continue
            st = numeric_stats(subset[col])
            low, high = schema["ranges"][col]
            value = sample_numeric_quantile_jitter(rng, st, col in schema["integer"])
            row[col] = int(np.clip(value, low, high)) if col in schema["integer"] else float(np.clip(value, low, high))
        rows.append(row)
    return normalize_to_schema(pd.DataFrame(rows), "pima", train_df)
def make_independent_fallback_rows(train_df, dataset, n, seed):
    return make_adult_fallback_rows(train_df, n, seed) if dataset == "adult" else make_pima_fallback_rows(train_df, n, seed)
def repair_or_refill_to_n(df, train_df, dataset, n, seed, reason, source_label):
    started = time.time()
    final, dedup = filter_valid_rows(df, train_df, dataset, True)
    initial_valid = int(len(final))
    attempts = 0
    while len(final) < n and attempts < CONFIG["max_refill_attempts"]:
        need = n - len(final)
        refill = make_independent_fallback_rows(train_df, dataset, max(need * 2, need), seed + attempts + 1000)
        final, more = filter_valid_rows(pd.concat([final, refill], ignore_index=True), train_df, dataset, True)
        dedup["removed_internal_duplicates"] += more["removed_internal_duplicates"]
        dedup["removed_train_duplicates"] += more["removed_train_duplicates"]
        attempts += 1
    final = final.head(n).reset_index(drop=True)
    primary_rows = int(min(initial_valid, len(final), n))
    repair_rows = int(max(0, len(final) - primary_rows))
    return final, {
        **validate_synthetic_data(final, train_df, dataset),
        **dedup,
        "initial_valid_rows": int(initial_valid),
        "final_valid_rows": int(len(final)),
        "primary_generator_rows": int(primary_rows),
        "fallback_or_refill_rows": int(repair_rows),
        "repair_rows": int(repair_rows),
        "repair_fraction": float(repair_rows / n) if n else np.nan,
        "repair_fallback_rows": int(repair_rows),
        "repair_fallback_fraction": float(repair_rows / n) if n else np.nan,
        "dedup_refill_rows": int(repair_rows),
        "refill_attempts": int(attempts),
        "repair_reason": reason if repair_rows else "",
        "repair_source": source_label if repair_rows else "",
        "repair_runtime_seconds": float(time.time() - started),
    }

def generate_fallback_only(train_df, dataset, n, seed):
    started = time.time()
    raw = make_independent_fallback_rows(train_df, dataset, n, seed)
    final, repair = repair_or_refill_to_n(raw, train_df, dataset, n, seed, "fallback_only_refill", "fallback_only")
    fallback_rows = int(len(final))
    return final, {
        **repair,
        "generation_status": "success" if len(final) == n else "failed",
        "effective_generator": "schema_independent_fallback",
        "method_success": bool(len(final) == n),
        "fallback_used": True,
        "fallback_reason": "fallback_only_baseline",
        "fallback_stage": "baseline",
        "fallback_rows": int(fallback_rows),
        "fallback_fraction": float(fallback_rows / n) if n else np.nan,
        "fallback_only_rows": int(fallback_rows),
        "raw_generated_rows": int(len(raw)),
        "raw_valid_rows": int(len(final)),
        "raw_valid_rate": float(len(final) / len(raw)) if len(raw) else 0.0,
        "primary_generator_rows": 0,
        "fallback_or_refill_rows": int(fallback_rows),
        "provenance_primary_source": "none",
        "provenance_fallback_source": "fallback_only",
        "generation_runtime_seconds": float(time.time() - started),
    }

def call_ollama(prompt, temperature, num_predict, timeout_seconds=None):
    global LAST_OLLAMA_CALL_METADATA
    started = time.time()
    url = CONFIG["ollama_endpoint"].rstrip("/") + "/api/generate"
    payload = {
        "model": CONFIG["ollama_model"],
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "top_p": CONFIG["ollama_top_p"], "num_predict": num_predict},
    }
    status_code = None
    response_bytes = 0
    try:
        response = requests.post(url, json=payload, timeout=timeout_seconds)
        status_code = response.status_code
        response_bytes = len(response.content or b"")
        response.raise_for_status()
        text = response.json().get("response", "")
        LAST_OLLAMA_CALL_METADATA = {
            "timestamp": utc_now_iso(),
            "endpoint": CONFIG.get("ollama_endpoint"),
            "model": CONFIG.get("ollama_model"),
            "temperature": float(temperature),
            "top_p": CONFIG.get("ollama_top_p"),
            "num_predict": int(num_predict),
            "timeout_seconds": timeout_seconds,
            "http_status": int(status_code) if status_code is not None else None,
            "elapsed_seconds": float(time.time() - started),
            "response_bytes": int(response_bytes),
            "ok": True,
            "error": "",
        }
        return text
    except Exception as exc:
        LAST_OLLAMA_CALL_METADATA = {
            "timestamp": utc_now_iso(),
            "endpoint": CONFIG.get("ollama_endpoint"),
            "model": CONFIG.get("ollama_model"),
            "temperature": float(temperature),
            "top_p": CONFIG.get("ollama_top_p"),
            "num_predict": int(num_predict),
            "timeout_seconds": timeout_seconds,
            "http_status": int(status_code) if status_code is not None else None,
            "elapsed_seconds": float(time.time() - started),
            "response_bytes": int(response_bytes),
            "ok": False,
            "error": f"{type(exc).__name__}: {exc}",
        }
        raise

## Direct-ICL-Prompts und CSV-Antworten verarbeiten


In [7]:
def strip_code_fences(text):
    return re.sub(r"```(?:python|csv)?|```", "", str(text), flags=re.IGNORECASE).strip()
def select_stratified_few_shot_examples(train_df, dataset, k, seed):
    target = schema_for(dataset)["target"]
    parts = []
    for _, group in train_df.groupby(target):
        take = min(len(group), max(1, math.ceil(k / max(train_df[target].nunique(), 1))))
        parts.append(group.sample(n=take, random_state=seed))
    return normalize_to_schema(pd.concat(parts).sample(frac=1, random_state=seed).head(k), dataset, train_df)
def target_distribution_text(train_df, dataset):
    target = schema_for(dataset)["target"]
    return "; ".join(f"{k}: {v:.3f}" for k, v in train_df[target].value_counts(normalize=True).sort_index().items())
def build_direct_icl_csv_prompt(train_df, dataset, batch_size, seed, attempt):
    examples = select_stratified_few_shot_examples(train_df, dataset, 8, seed + attempt)
    buffer = io.StringIO()
    examples.to_csv(buffer, index=False)
    columns = ",".join(schema_for(dataset)["columns"])
    return "\n".join([
        f"Generate {batch_size} synthetic CSV rows for the {dataset} dataset.",
        "Return only CSV text with one header line. No markdown. No explanation.",
        f"Columns in this exact order: {columns}",
        f"Approximate target distribution: {target_distribution_text(train_df, dataset)}",
        "Do not copy example rows exactly. Vary numeric values and category combinations.",
        "Examples:",
        buffer.getvalue().strip(),
    ])
def parse_direct_icl_csv_response(text, columns):
    cleaned = strip_code_fences(text)
    rows, invalid, errors, raw_rows = [], 0, 0, 0
    invalid_lines = []
    invalid_reason_counts = {"wrong_column_count": 0, "parse_error": 0}
    try:
        for row in csv.reader(io.StringIO(cleaned)):
            row = [str(x).strip() for x in row]
            if not row or all(x == "" for x in row):
                continue
            if row == columns:
                continue
            raw_rows += 1
            if len(row) != len(columns):
                invalid += 1
                invalid_reason_counts["wrong_column_count"] += 1
                if len(invalid_lines) < 10:
                    invalid_lines.append({"reason": "wrong_column_count", "columns_seen": len(row), "line_preview": ",".join(row)[:500]})
                continue
            rows.append(row)
    except Exception as exc:
        errors += 1
        invalid_reason_counts["parse_error"] += 1
        if len(invalid_lines) < 10:
            invalid_lines.append({"reason": f"parse_error: {type(exc).__name__}: {exc}", "line_preview": ""})
    return pd.DataFrame(rows, columns=columns), {
        "invalid_format_rows": invalid,
        "parse_error_count": errors,
        "raw_rows": int(raw_rows),
        "parsed_rows": int(len(rows)),
        "invalid_lines": invalid_lines,
        "invalid_reason_counts": invalid_reason_counts,
    }
def generate_direct_icl(train_df, dataset, n, seed, audit_context=None, train_size=None):
    started = time.time()
    columns = schema_for(dataset)["columns"]
    frames, llm_calls, parse_errors, invalid_format = [], 0, 0, 0
    train_size = int(train_size) if train_size is not None else int(len(train_df))
    for attempt in range(CONFIG["max_direct_icl_batches"]):
        if sum(len(x) for x in frames) >= n:
            break
        prompt = build_direct_icl_csv_prompt(train_df, dataset, CONFIG["direct_icl_batch_size"], seed, attempt)
        ollama_call_metadata = {}
        try:
            response = call_ollama(prompt, CONFIG["direct_icl_temperature"], CONFIG["direct_icl_num_predict"], timeout_seconds=CONFIG.get("ollama_timeout_seconds"))
            ollama_call_metadata = dict(globals().get("LAST_OLLAMA_CALL_METADATA", {}))
            llm_calls += 1
        except Exception as exc:
            response = ""
            ollama_call_metadata = dict(globals().get("LAST_OLLAMA_CALL_METADATA", {}))
            parse_errors += 1
            batch_error = f"{type(exc).__name__}: {exc}"
        else:
            batch_error = ""
        parsed, diag = parse_direct_icl_csv_response(response, columns)
        valid_batch, _ = filter_valid_rows(parsed, train_df, dataset, True) if len(parsed) else (pd.DataFrame(columns=columns), {})
        diag = {**diag, "valid_rows": int(len(valid_batch)), "batch_error": batch_error, "ollama_call_metadata": ollama_call_metadata}
        save_direct_icl_batch_log(audit_context, dataset, train_size, seed, attempt, prompt, response, diag)
        parse_errors += diag["parse_error_count"]
        invalid_format += diag["invalid_format_rows"]
        if len(parsed):
            frames.append(parsed)
    raw = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
    valid_llm, _ = filter_valid_rows(raw, train_df, dataset, True)
    primary_only = valid_llm.head(n).reset_index(drop=True)
    llm_valid_rows = int(len(primary_only))
    final, repair = repair_or_refill_to_n(valid_llm, train_df, dataset, n, seed, "direct_icl_refill", "direct_icl_repair_fallback")
    fallback_rows = max(0, n - llm_valid_rows)
    return final, {
        **repair,
        "generation_status": "success",
        "effective_generator": "llm_direct_rows" if llm_valid_rows else "schema_independent_fallback",
        "method_success": bool(llm_valid_rows >= max(1, int(CONFIG["method_success_min_llm_fraction"] * n))),
        "fallback_used": bool(fallback_rows > 0),
        "fallback_reason": "insufficient_valid_direct_icl_rows" if fallback_rows else "",
        "fallback_stage": "validation_refill" if fallback_rows else "",
        "fallback_rows": int(fallback_rows),
        "fallback_fraction": float(fallback_rows / n),
        "llm_raw_rows": int(len(raw)),
        "llm_valid_rows": int(llm_valid_rows),
        "llm_valid_fraction": float(llm_valid_rows / len(raw)) if len(raw) else 0.0,
        "primary_only_rows": int(llm_valid_rows),
        "primary_only_fraction_of_target": float(llm_valid_rows / n) if n else np.nan,
        "parse_error_count": int(parse_errors),
        "invalid_format_rows": int(invalid_format),
        "llm_calls": int(llm_calls),
        "llm_attempts": int(llm_calls),
        "llm_repair_attempts": 0,
        "ollama_model": CONFIG["ollama_model"],
        "ollama_temperature": CONFIG["direct_icl_temperature"],
        "ollama_top_p": CONFIG["ollama_top_p"],
        "ollama_num_predict": CONFIG["direct_icl_num_predict"],
        "provenance_primary_source": "direct_icl_primary_valid",
        "provenance_fallback_source": "direct_icl_repair_fallback",
        "generation_runtime_seconds": float(time.time() - started),
        "_diagnostic_frames": {"primary_valid_only": primary_only},
    }

def prepare_ctgan_training_frame(train_df, dataset):
    frame = normalize_to_schema(train_df, dataset, train_df)
    return frame.drop(columns=["education-num"]) if dataset == "adult" else frame
def restore_ctgan_schema(sampled_df, train_df, dataset):
    frame = pd.DataFrame(sampled_df).copy()
    if dataset == "adult" and "education-num" not in frame.columns and "education" in frame.columns:
        frame["education-num"] = frame["education"].map(EDUCATION_TO_NUM)
    return normalize_to_schema(frame, dataset, train_df)
def fit_ctgan(train_df, dataset, epochs=None):
    if SingleTableMetadata is None or CTGANSynthesizer is None:
        raise ImportError("sdv is required for CTGAN")
    frame = prepare_ctgan_training_frame(train_df, dataset)
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(frame)
    epochs = int(CONFIG["ctgan_epochs"] if epochs is None else epochs)
    synth = CTGANSynthesizer(metadata, epochs=epochs, verbose=False)
    synth.fit(frame)
    return synth
def generate_ctgan(train_df, dataset, n, seed, epochs=None, method_label="ctgan"):
    started = time.time()
    set_all_seeds(seed)
    raw_frames, raw_sampled, method_success, fallback_reason = [], 0, True, ""
    try:
        ctgan_epochs_used = int(CONFIG["ctgan_epochs"] if epochs is None else epochs)
        synth = fit_ctgan(train_df, dataset, epochs=ctgan_epochs_used)
        while raw_sampled < n * CONFIG["ctgan_max_raw_multiplier"]:
            draw = n * CONFIG["ctgan_raw_multiplier"]
            raw = synth.sample(draw)
            raw_frames.append(raw)
            raw_sampled += len(raw)
            valid, _ = filter_valid_rows(restore_ctgan_schema(pd.concat(raw_frames, ignore_index=True), train_df, dataset), train_df, dataset, True)
            if len(valid) >= n:
                break
        raw_all = pd.concat(raw_frames, ignore_index=True)
    except Exception as exc:
        raw_all = pd.DataFrame()
        raw_sampled = 0
        method_success = False
        fallback_reason = f"ctgan_failed: {type(exc).__name__}: {exc}"
    restored = restore_ctgan_schema(raw_all, train_df, dataset) if len(raw_all) else pd.DataFrame(columns=schema_for(dataset)["columns"])
    raw_valid, _ = filter_valid_rows(restored, train_df, dataset, True)
    raw_valid_rows = int(len(raw_valid))
    final, repair = repair_or_refill_to_n(raw_valid, train_df, dataset, n, seed, fallback_reason or "ctgan_refill", "ctgan_postprocess_fallback")
    fallback_rows = max(0, n - min(raw_valid_rows, n))
    ctgan_epochs_used = int(CONFIG["ctgan_epochs"] if epochs is None else epochs)
    return final, {
        **repair,
        "generation_status": "success" if len(final) == n else "failed",
        "effective_generator": method_label if method_success and raw_valid_rows else "schema_independent_fallback",
        "method_success": bool(method_success and raw_valid_rows >= max(1, int(0.5 * n))),
        "fallback_used": bool(fallback_rows > 0 or not method_success),
        "fallback_reason": fallback_reason if fallback_rows or not method_success else "",
        "fallback_stage": "ctgan_sampling_or_validation" if fallback_rows or not method_success else "",
        "fallback_rows": int(fallback_rows),
        "fallback_fraction": float(fallback_rows / n),
        "ctgan_epochs_used": int(ctgan_epochs_used),
        "ctgan_raw_sampled_rows": int(raw_sampled),
        "ctgan_raw_valid_rows": int(raw_valid_rows),
        "ctgan_raw_valid_rate": float(raw_valid_rows / raw_sampled) if raw_sampled else 0.0,
        "ctgan_rejection_rows": int(max(0, raw_sampled - raw_valid_rows)),
        "ctgan_repair_rows": int(fallback_rows),
        "ctgan_repair_fraction": float(fallback_rows / n),
        "postprocessing_used": True,
        "postprocessing_type": "schema_normalization" if method_success else "full_schema_independent_fallback",
        "provenance_primary_source": method_label,
        "provenance_fallback_source": f"{method_label}_postprocess_fallback",
        "raw_valid_rows": int(raw_valid_rows),
        "raw_valid_rate": float(raw_valid_rows / raw_sampled) if raw_sampled else 0.0,
        "final_valid_rate": float(len(final) / n) if n else np.nan,
        "llm_calls": 0,
        "llm_attempts": 0,
        "llm_repair_attempts": 0,
        "generation_runtime_seconds": float(time.time() - started),
    }

## Code-LLM-v2 Generator vorbereiten


In [8]:
def build_code_llm_body_prompt(dataset, train_df, stats):
    schema = schema_for(dataset)
    if dataset == "adult":
        example_body = """
for _ in range(n):
    education = sample_categorical(rng, CATEGORICAL_STATS["education"])
    rows.append({
        "age": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["age"], True),
        "workclass": sample_categorical(rng, CATEGORICAL_STATS["workclass"]),
        "education": education,
        "education-num": EDUCATION_TO_NUM.get(education, 9),
        "marital-status": sample_categorical(rng, CATEGORICAL_STATS["marital-status"]),
        "occupation": sample_categorical(rng, CATEGORICAL_STATS["occupation"]),
        "race": sample_categorical(rng, CATEGORICAL_STATS["race"]),
        "sex": sample_categorical(rng, CATEGORICAL_STATS["sex"]),
        "hours-per-week": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["hours-per-week"], True),
        "income": sample_categorical(rng, TARGET_STATS),
    })
""".strip()
    else:
        numeric_lines = []
        for col in schema["columns"]:
            if col == schema["target"]:
                numeric_lines.append(f'        "{col}": int(sample_categorical(rng, TARGET_STATS)),')
            else:
                numeric_lines.append(f'        "{col}": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["{col}"], {col in schema["integer"]}),')
        example_body = "for _ in range(n):\n    rows.append({\n" + "\n".join(numeric_lines) + "\n    })"
    return "\n".join([
        "Complete only the Python body inside generate_synthetic_data(n, seed).",
        f"This is {CONFIG.get('code_llm_scaffold_version', 'Code-LLM-v2')}: use the provided API exactly.",
        "Return only executable Python body text, no markdown, no imports, no function definition.",
        "Available variables: rng, rows, COLUMNS, COLUMN_STATS, NUMERIC_STATS, CATEGORICAL_STATS, TARGET_STATS, ALLOWED_VALUES, EDUCATION_TO_NUM.",
        "Stats layout: COLUMN_STATS['numeric'][col]['min'], ['max'], ['mean'], ['std'], ['q05'], ['q25'], ['q50'], ['q75'], ['q95']; COLUMN_STATS['categorical'][col]['values']/['probs']; COLUMN_STATS['target_probs']['values']/['probs'].",
        "Prefer the aliases NUMERIC_STATS[col], CATEGORICAL_STATS[col], and TARGET_STATS instead of direct COLUMN_STATS access.",
        "Preferred numeric API: NUMERIC_STATS[col] with keys min, max, mean, std, q05, q25, q50, q75, q95.",
        "Preferred categorical API: CATEGORICAL_STATS[col] and TARGET_STATS, each with keys values and probs.",
        "Allowed helper calls:",
        "sample_numeric_quantile_jitter(rng, NUMERIC_STATS[col], integer=True_or_False)",
        "sample_truncated_normal(rng, NUMERIC_STATS[col], integer=True_or_False) or sample_truncated_normal(rng, stats['mean'], stats['std'], stats['min'], stats['max'], integer=True_or_False)",
        "sample_categorical(rng, CATEGORICAL_STATS[col]) or sample_categorical(rng, CATEGORICAL_STATS[col]['values'], CATEGORICAL_STATS[col]['probs'])",
        "For the target use sample_categorical(rng, TARGET_STATS).",
        "Do not use ALLOWED_VALUES['values'], ALLOWED_VALUES[\"values\"], ALLOWED_VALUES['probs'], or ALLOWED_VALUES[\"probs\"].",
        "Do not use stats['low'], stats[\"low\"], stats['high'], stats[\"high\"], COLUMN_STATS[col], rng.truncnorm, free helper names, external files, open, eval, exec, or imports.",
        "Do not create custom probability arrays unless they are non-negative, finite, length-matched, and normalized; use provided stats instead.",
        "Append dictionaries to rows and generate at least n rows. The scaffold returns pd.DataFrame(rows[:n], columns=COLUMNS).",
        f"Dataset: {dataset}",
        f"Columns: {schema['columns']}",
        f"Allowed values: {schema['allowed_values']}",
        f"Column statistics: {stats}",
        "Valid minimal body example:",
        example_body,
    ])
def extract_code_body(text):
    cleaned = strip_code_fences(text)
    if "def generate_synthetic_data" in cleaned:
        try:
            module = ast.parse(cleaned)
            for node in module.body:
                if isinstance(node, ast.FunctionDef) and node.name == "generate_synthetic_data" and node.body:
                    lines = cleaned.splitlines()
                    body = lines[node.body[0].lineno - 1: node.end_lineno]
                    return "\n".join(re.sub(r"^    ", "", line) for line in body).strip()
        except Exception:
            pass
    lines = []
    started = False
    for line in cleaned.splitlines():
        stripped = line.strip()
        if not stripped:
            if started:
                lines.append(line)
            continue
        if not started and stripped.lower().startswith(("here", "sure", "this")):
            continue
        if stripped.startswith("return pd.DataFrame"):
            continue
        if not started and any(token in stripped for token in ["rows.append", "for ", "if ", "=", "while "]):
            started = True
        if started:
            lines.append(line)
    return "\n".join(lines).strip()
def sanitize_code_body(text):
    body = extract_code_body(text)
    body = body.replace("rng.clip(", "np.clip(").replace(".clamp(", ".clip(")
    return body.strip()
def wrap_code_body_in_scaffold(body):
    body = sanitize_code_body(body) or "for _ in range(n):\n    rows.append({col: np.nan for col in COLUMNS})"
    indented = "\n".join(("    " + line if line.strip() else "") for line in body.splitlines())
    return "\n".join([
        "def generate_synthetic_data(n: int, seed: int) -> pd.DataFrame:",
        "    rng = np.random.default_rng(seed)",
        "    rows = []",
        indented,
        "    return pd.DataFrame(rows[:n], columns=COLUMNS)",
        "",
    ])
def compile_generated_code(code):
    ast.parse(code)
    return compile(code, "<code_llm_generated>", "exec")
def static_code_llm_contract_check(body):
    errors = []
    text = body or ""
    forbidden_patterns = [
        ("ALLOWED_VALUES['values']", "Do not use ALLOWED_VALUES['values']; use TARGET_STATS or CATEGORICAL_STATS[col]."),
        ('ALLOWED_VALUES["values"]', "Do not use ALLOWED_VALUES['values']; use TARGET_STATS or CATEGORICAL_STATS[col]."),
        ("ALLOWED_VALUES['probs']", "Do not use ALLOWED_VALUES['probs']; use TARGET_STATS or CATEGORICAL_STATS[col]."),
        ('ALLOWED_VALUES["probs"]', "Do not use ALLOWED_VALUES['probs']; use TARGET_STATS or CATEGORICAL_STATS[col]."),
        ("['low']", "Do not use ['low']; numeric stats use ['min']."),
        ('["low"]', "Do not use ['low']; numeric stats use ['min']."),
        ("['high']", "Do not use ['high']; numeric stats use ['max']."),
        ('["high"]', "Do not use ['high']; numeric stats use ['max']."),
        (".truncnorm", "Do not use rng.truncnorm or scipy truncnorm; use sample_truncated_normal or sample_numeric_quantile_jitter."),
        ("open(", "Do not access files from generated sampler code."),
        ("eval(", "Do not use eval."),
        ("exec(", "Do not use exec."),
        ("__import__", "Do not import modules."),
        ("compile(", "Do not compile code inside generated sampler code."),
        ("input(", "Do not use input."),
    ]
    for pattern, message in forbidden_patterns:
        if pattern in text:
            errors.append(message)
    try:
        module = ast.parse(text if text.strip() else "pass")
    except SyntaxError as exc:
        return [f"SyntaxError before scaffold compile: {exc}"]
    for node in ast.walk(module):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            errors.append("Do not write imports; use only the scaffold namespace.")
        if isinstance(node, ast.Return):
            errors.append("Do not return from the body; append dictionaries to rows only.")
        if isinstance(node, ast.Call):
            func = node.func
            if isinstance(func, ast.Name) and func.id in {"open", "eval", "exec", "compile", "input", "__import__"}:
                errors.append(f"Forbidden call: {func.id}().")
            if isinstance(func, ast.Attribute) and func.attr in {"read_csv", "to_csv", "to_excel", "to_parquet"}:
                errors.append(f"Do not use file IO: {func.attr}().")
            if isinstance(func, ast.Name) and func.id.startswith("sample_") and func.id not in {"sample_categorical", "sample_numeric_quantile_jitter", "sample_truncated_normal"}:
                errors.append(f"Unknown helper {func.id}; use only the documented helper functions.")
    return sorted(set(errors))

def safe_code_namespace(dataset, stats):
    flat_stats = {}
    flat_stats.update(stats.get("numeric", {}))
    flat_stats.update(stats.get("categorical", {}))
    flat_stats[schema_for(dataset)["target"]] = stats.get("target_probs", {})
    safe_builtins = {
        "abs": abs, "bool": bool, "dict": dict, "enumerate": enumerate, "float": float,
        "int": int, "isinstance": isinstance, "len": len, "list": list, "max": max,
        "min": min, "range": range, "round": round, "str": str, "sum": sum, "tuple": tuple, "zip": zip,
    }
    return {
        "__builtins__": safe_builtins,
        "pd": pd,
        "np": np,
        "math": math,
        "random": random,
        "COLUMNS": schema_for(dataset)["columns"],
        "COLUMN_STATS": stats,
        "FLAT_COLUMN_STATS": flat_stats,
        "NUMERIC_STATS": stats.get("numeric", {}),
        "CATEGORICAL_STATS": stats.get("categorical", {}),
        "TARGET_STATS": stats.get("target_probs", {}),
        "ALLOWED_VALUES": schema_for(dataset)["allowed_values"],
        "EDUCATION_TO_NUM": EDUCATION_TO_NUM,
        "sample_categorical": sample_categorical,
        "sample_numeric_quantile_jitter": sample_numeric_quantile_jitter,
        "sample_truncated_normal": sample_truncated_normal,
    }
def run_generated_sampler(code_text, dataset, stats, n, seed):
    namespace = safe_code_namespace(dataset, stats)
    exec(compile_generated_code(code_text), namespace)
    if "generate_synthetic_data" not in namespace:
        raise NameError("generate_synthetic_data is missing")
    return normalize_to_schema(namespace["generate_synthetic_data"](n, seed), dataset)
def build_code_repair_prompt(body, stage, message, dataset, stats):
    schema = schema_for(dataset)
    mini = """
for _ in range(n):
    rows.append({
        COLUMNS[0]: sample_numeric_quantile_jitter(rng, NUMERIC_STATS[COLUMNS[0]], COLUMNS[0] in INTEGER_COLUMNS),
    })
""".strip()
    if dataset == "adult":
        mini = """
for _ in range(n):
    education = sample_categorical(rng, CATEGORICAL_STATS["education"])
    rows.append({
        "age": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["age"], True),
        "workclass": sample_categorical(rng, CATEGORICAL_STATS["workclass"]),
        "education": education,
        "education-num": EDUCATION_TO_NUM.get(education, 9),
        "marital-status": sample_categorical(rng, CATEGORICAL_STATS["marital-status"]),
        "occupation": sample_categorical(rng, CATEGORICAL_STATS["occupation"]),
        "race": sample_categorical(rng, CATEGORICAL_STATS["race"]),
        "sex": sample_categorical(rng, CATEGORICAL_STATS["sex"]),
        "hours-per-week": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["hours-per-week"], True),
        "income": sample_categorical(rng, TARGET_STATS),
    })
""".strip()
    elif dataset == "pima":
        mini = """
for _ in range(n):
    rows.append({
        "Pregnancies": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["Pregnancies"], True),
        "Glucose": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["Glucose"], True),
        "BloodPressure": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["BloodPressure"], True),
        "SkinThickness": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["SkinThickness"], True),
        "Insulin": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["Insulin"], True),
        "BMI": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["BMI"], False),
        "DiabetesPedigreeFunction": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["DiabetesPedigreeFunction"], False),
        "Age": sample_numeric_quantile_jitter(rng, NUMERIC_STATS["Age"], True),
        "Outcome": int(sample_categorical(rng, TARGET_STATS)),
    })
""".strip()
    return "\n".join([
        "Repair only the Python body below. Return only corrected Python body; no markdown, no imports, no function definition.",
        f"Scaffold version: {CONFIG.get('code_llm_scaffold_version', 'Code-LLM-v2')}",
        f"Error stage: {stage}",
        f"Error message: {message}",
        "Do not repeat the same failing code. Rewrite using the exact API rules below.",
        "For TypeError in helper calls: use sample_categorical(rng, stats_dict) or sample_categorical(rng, values, probs); use sample_truncated_normal(rng, stats_dict, integer=...) or pass mean, std, min, max.",
        "For KeyError low/high/values/probs: numeric stats have min/max, categorical and target stats have values/probs; ALLOWED_VALUES has only per-column allowed category lists.",
        "For AttributeError truncnorm: rng has no truncnorm; use sample_truncated_normal or sample_numeric_quantile_jitter.",
        "For SyntaxError: output plain indented body statements only, no fences or prose.",
        "Never use ALLOWED_VALUES['values'], ALLOWED_VALUES['probs'], stats['low'], stats['high'], COLUMN_STATS[col], rng.truncnorm, imports, open, eval, or exec.",
        "For categorical columns use CATEGORICAL_STATS[col]['values'] and ['probs'] or CATEGORICAL_STATS[col].",
        "For the target use TARGET_STATS['values'] and ['probs'] or TARGET_STATS.",
        "The body must append dictionaries to rows and must not return a DataFrame.",
        f"Dataset: {dataset}",
        f"Columns: {schema['columns']}",
        f"Allowed values: {schema['allowed_values']}",
        f"Column statistics: {stats}",
        "Valid body pattern:",
        mini,
        "Current failing body:",
        body,
    ])
def standalone_helper_code(dataset, stats, metadata=None):
    schema = schema_for(dataset)
    flat_stats = {}
    flat_stats.update(stats.get("numeric", {}))
    flat_stats.update(stats.get("categorical", {}))
    flat_stats[schema["target"]] = stats.get("target_probs", {})
    lines = [
        "import numpy as np",
        "import pandas as pd",
        "import math",
        "import random",
        "",
        f"COLUMNS = {repr(schema['columns'])}",
        f"COLUMN_STATS = {repr(stats)}",
        "NUMERIC_STATS = COLUMN_STATS.get('numeric', {})",
        "CATEGORICAL_STATS = COLUMN_STATS.get('categorical', {})",
        "TARGET_STATS = COLUMN_STATS.get('target_probs', {})",
        f"FLAT_COLUMN_STATS = {repr(flat_stats)}",
        f"ALLOWED_VALUES = {repr(schema['allowed_values'])}",
        f"EDUCATION_TO_NUM = {repr(EDUCATION_TO_NUM)}",
    ]
    if metadata is not None:
        lines.append(f"CODE_LLM_METADATA = {repr(metadata)}")
    lines.extend([
        "",
        "def sample_categorical(rng, values, probs=None):",
        "    if isinstance(values, dict):",
        "        probs = values.get('probs', probs)",
        "        values = values.get('values', [])",
        "    if isinstance(values, str) or not hasattr(values, '__iter__'):",
        "        values = [values]",
        "    values = list(values) if values is not None else []",
        "    if not values:",
        "        return np.nan",
        "    try:",
        "        p = np.asarray(probs, dtype=float) if probs is not None else np.ones(len(values), dtype=float)",
        "    except Exception:",
        "        p = np.ones(len(values), dtype=float)",
        "    if p.ndim != 1 or len(p) != len(values) or not np.all(np.isfinite(p)) or np.any(p < 0) or float(p.sum()) <= 0.0:",
        "        p = np.ones(len(values), dtype=float)",
        "    p = p / p.sum()",
        "    return values[int(rng.choice(len(values), p=p))]",
        "",
        "def sample_numeric_quantile_jitter(rng, stats, integer=False):",
        "    anchors = np.array([stats['q05'], stats['q25'], stats['q50'], stats['q75'], stats['q95']], dtype=float)",
        "    value = float(rng.choice(anchors))",
        "    value += float(rng.normal(0.0, max((stats['q95'] - stats['q05']) / 12.0, 1e-6)))",
        "    value = float(np.clip(value, stats['min'], stats['max']))",
        "    return int(round(value)) if integer else value",
        "",
        "def sample_truncated_normal(rng, mean, std=None, low=None, high=None, integer=False):",
        "    if isinstance(mean, dict):",
        "        stats = mean",
        "        mean = stats.get('mean', stats.get('q50', 0.0))",
        "        std = stats.get('std', std)",
        "        if std is None:",
        "            std = max((float(stats.get('q95', mean)) - float(stats.get('q05', mean))) / 4.0, 1e-6)",
        "        low = stats.get('low', stats.get('min', stats.get('q05', low)))",
        "        high = stats.get('high', stats.get('max', stats.get('q95', high)))",
        "    mean = float(0.0 if mean is None or pd.isna(mean) else mean)",
        "    std = float(1e-6 if std is None or pd.isna(std) else std)",
        "    std = max(std, 1e-6)",
        "    if low is None or pd.isna(low):",
        "        low = mean - 4.0 * std",
        "    if high is None or pd.isna(high):",
        "        high = mean + 4.0 * std",
        "    low, high = float(low), float(high)",
        "    if low > high:",
        "        low, high = high, low",
        "    value = float(np.clip(rng.normal(mean, std), low, high))",
        "    return int(round(value)) if integer else value",
        "",
    ])
    return "\n".join(lines)
def fallback_program_code(dataset, stats, reason):
    lines = [
        "# Fallback sampler used because LLM-generated code failed.",
        f"# Original failure reason: {reason}",
        "",
        standalone_helper_code(dataset, stats),
        "",
        "def generate_synthetic_data(n: int, seed: int) -> pd.DataFrame:",
        "    rng = np.random.default_rng(seed)",
        "    rows = []",
        "    for _ in range(n):",
    ]
    if dataset == "adult":
        lines.extend([
            "        education = sample_categorical(rng, COLUMN_STATS['categorical']['education']['values'], COLUMN_STATS['categorical']['education']['probs'])",
            "        rows.append({",
            "            'age': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['age'], True),",
            "            'workclass': sample_categorical(rng, COLUMN_STATS['categorical']['workclass']['values'], COLUMN_STATS['categorical']['workclass']['probs']),",
            "            'education': education,",
            "            'education-num': EDUCATION_TO_NUM.get(education, 9),",
            "            'marital-status': sample_categorical(rng, COLUMN_STATS['categorical']['marital-status']['values'], COLUMN_STATS['categorical']['marital-status']['probs']),",
            "            'occupation': sample_categorical(rng, COLUMN_STATS['categorical']['occupation']['values'], COLUMN_STATS['categorical']['occupation']['probs']),",
            "            'race': sample_categorical(rng, COLUMN_STATS['categorical']['race']['values'], COLUMN_STATS['categorical']['race']['probs']),",
            "            'sex': sample_categorical(rng, COLUMN_STATS['categorical']['sex']['values'], COLUMN_STATS['categorical']['sex']['probs']),",
            "            'hours-per-week': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['hours-per-week'], True),",
            "            'income': sample_categorical(rng, COLUMN_STATS['target_probs']['values'], COLUMN_STATS['target_probs']['probs']),",
            "        })",
        ])
    else:
        lines.extend([
            "        rows.append({",
            "            'Pregnancies': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['Pregnancies'], True),",
            "            'Glucose': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['Glucose'], True),",
            "            'BloodPressure': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['BloodPressure'], True),",
            "            'SkinThickness': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['SkinThickness'], True),",
            "            'Insulin': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['Insulin'], True),",
            "            'BMI': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['BMI'], False),",
            "            'DiabetesPedigreeFunction': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['DiabetesPedigreeFunction'], False),",
            "            'Age': sample_numeric_quantile_jitter(rng, COLUMN_STATS['numeric']['Age'], True),",
            "            'Outcome': int(sample_categorical(rng, COLUMN_STATS['target_probs']['values'], COLUMN_STATS['target_probs']['probs'])),",
            "        })",
        ])
    lines.extend(["    return pd.DataFrame(rows, columns=COLUMNS)", ""])
    return "\n".join(lines)
def save_code_llm_program(path, code_text):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(code_text, encoding="utf-8")
def generate_code_llm(train_df, dataset, n, seed, code_path, audit_context=None, train_size=None):
    started = time.time()
    stats = get_column_statistics(train_df, dataset)
    train_size = int(train_size) if train_size is not None else int(len(train_df))
    body, function_code, failure_stage, failure_reason = "", "", "", ""
    generation_success = compile_success = runtime_success = validation_success = repair_success = False
    llm_calls = 0
    raw_df = pd.DataFrame(columns=schema_for(dataset)["columns"])
    for attempt in range(CONFIG["max_code_llm_attempts"]):
        trace = {
            "prompt": "",
            "raw_response": "",
            "sanitized_code_body": "",
            "wrapped_scaffold_code": "",
            "response_status": "not_started",
            "sanitize_status": "not_started",
            "static_contract_status": "not_started",
            "static_contract_errors": [],
            "compile_status": "not_started",
            "runtime_smoke_status": "not_started",
            "smoke_validation_status": "not_started",
            "runtime_full_status": "not_started",
            "full_validation_status": "not_started",
            "failure_stage": "",
            "failure_reason": "",
            "ollama_call_metadata": {},
            "http_status": None,
            "request_elapsed_seconds": None,
        }
        try:
            prompt = build_code_llm_body_prompt(dataset, train_df, stats) if attempt == 0 else build_code_repair_prompt(body, failure_stage, failure_reason, dataset, stats)
            trace["prompt"] = prompt
            try:
                response = call_ollama(prompt, CONFIG["code_llm_temperature"], CONFIG["code_llm_num_predict"], timeout_seconds=CONFIG.get("ollama_timeout_seconds"))
                trace["ollama_call_metadata"] = dict(globals().get("LAST_OLLAMA_CALL_METADATA", {}))
                trace["http_status"] = trace["ollama_call_metadata"].get("http_status")
                trace["request_elapsed_seconds"] = trace["ollama_call_metadata"].get("elapsed_seconds")
                trace["raw_response"] = response
                trace["response_status"] = "success"
                llm_calls += 1
            except Exception as exc:
                trace["ollama_call_metadata"] = dict(globals().get("LAST_OLLAMA_CALL_METADATA", {}))
                trace["http_status"] = trace["ollama_call_metadata"].get("http_status")
                trace["request_elapsed_seconds"] = trace["ollama_call_metadata"].get("elapsed_seconds")
                failure_stage = "response"
                failure_reason = f"{type(exc).__name__}: {exc}"
                trace["response_status"] = "failed"
                raise
            try:
                body = sanitize_code_body(response)
                trace["sanitized_code_body"] = body
                trace["sanitize_status"] = "success" if body else "empty"
            except Exception as exc:
                failure_stage = "sanitize"
                failure_reason = f"{type(exc).__name__}: {exc}"
                trace["sanitize_status"] = "failed"
                raise
            try:
                contract_errors = static_code_llm_contract_check(body)
                trace["static_contract_errors"] = contract_errors
                if contract_errors:
                    failure_stage = "static_contract"
                    failure_reason = "; ".join(contract_errors)
                    trace["static_contract_status"] = "failed"
                    raise ValueError(failure_reason)
                trace["static_contract_status"] = "success"
                function_code = wrap_code_body_in_scaffold(body)
                trace["wrapped_scaffold_code"] = function_code
                compile_generated_code(function_code)
                compile_success = True
                trace["compile_status"] = "success"
            except Exception as exc:
                if failure_stage == "static_contract":
                    trace["compile_status"] = "skipped_static_contract"
                else:
                    failure_stage = "compile"
                    failure_reason = f"{type(exc).__name__}: {exc}"
                    trace["compile_status"] = "failed"
                raise
            try:
                smoke = run_generated_sampler(function_code, dataset, stats, min(25, n), seed)
                runtime_success = True
                trace["runtime_smoke_status"] = "success"
                trace["smoke_rows"] = int(len(smoke))
            except Exception as exc:
                failure_stage = "runtime_smoke"
                failure_reason = f"{type(exc).__name__}: {exc}"
                trace["runtime_smoke_status"] = "failed"
                raise
            try:
                smoke_valid, _ = filter_valid_rows(smoke, train_df, dataset, True)
                trace["smoke_valid_rows"] = int(len(smoke_valid))
                trace["smoke_validation_status"] = "success" if len(smoke_valid) else "failed"
                if len(smoke_valid) == 0:
                    raise ValueError("smoke output produced no valid rows")
            except Exception as exc:
                failure_stage = "smoke_validation"
                failure_reason = f"{type(exc).__name__}: {exc}"
                trace["smoke_validation_status"] = "failed"
                raise
            try:
                raw_df = run_generated_sampler(function_code, dataset, stats, n, seed)
                trace["runtime_full_status"] = "success"
                trace["full_rows"] = int(len(raw_df))
            except Exception as exc:
                failure_stage = "runtime_full"
                failure_reason = f"{type(exc).__name__}: {exc}"
                trace["runtime_full_status"] = "failed"
                raise
            try:
                valid_llm, _ = filter_valid_rows(raw_df, train_df, dataset, True)
                validation_success = len(valid_llm) > 0
                trace["full_valid_rows"] = int(len(valid_llm))
                trace["full_validation_status"] = "success" if validation_success else "failed"
                if not validation_success:
                    failure_stage = "full_validation"
                    failure_reason = "full generation produced no valid rows"
            except Exception as exc:
                failure_stage = "full_validation"
                failure_reason = f"{type(exc).__name__}: {exc}"
                trace["full_validation_status"] = "failed"
                raise
            generation_success = True
            repair_success = attempt > 0
            break
        except Exception as exc:
            if not failure_stage:
                failure_stage = "response"
                failure_reason = f"{type(exc).__name__}: {exc}"
        finally:
            trace["failure_stage"] = failure_stage if trace.get("failure_stage", "") == "" else trace["failure_stage"]
            trace["failure_reason"] = failure_reason if trace.get("failure_reason", "") == "" else trace["failure_reason"]
            trace["model"] = CONFIG["ollama_model"]
            trace["temperature"] = CONFIG["code_llm_temperature"]
            trace["top_p"] = CONFIG["ollama_top_p"]
            trace["num_predict"] = CONFIG["code_llm_num_predict"]
            save_code_llm_attempt_trace(audit_context, dataset, train_size, seed, attempt, trace)
    valid_llm, _ = filter_valid_rows(raw_df, train_df, dataset, True)
    llm_valid_rows = int(min(len(valid_llm), n))
    final, repair = repair_or_refill_to_n(valid_llm, train_df, dataset, n, seed, failure_reason or "code_llm_refill", "code_llm_fallback")
    fallback_rows = max(0, n - llm_valid_rows)
    if fallback_rows:
        save_code_llm_fallback_completion_trace(audit_context, dataset, train_size, seed, len(final), fallback_rows, failure_stage, failure_reason)
    metadata = {"dataset": dataset, "seed": int(seed), "scaffold_version": CONFIG.get("code_llm_scaffold_version", "Code-LLM-v2"), "generation_success": generation_success, "compile_success": compile_success, "runtime_success": runtime_success, "validation_success": validation_success, "fallback_used": bool(fallback_rows), "failure_stage": failure_stage, "failure_reason": failure_reason, "llm_attempts": int(llm_calls)}
    if generation_success:
        save_code_llm_program(code_path, "\n\n".join([standalone_helper_code(dataset, stats, metadata), function_code]))
    else:
        save_code_llm_program(code_path, fallback_program_code(dataset, stats, failure_reason))
    return final, {
        **repair,
        "generation_status": "success",
        "effective_generator": "llm_generated_code" if llm_valid_rows else "schema_independent_fallback",
        "method_success": bool(llm_valid_rows >= max(1, int(CONFIG["method_success_min_llm_fraction"] * n))),
        "fallback_used": bool(fallback_rows),
        "fallback_reason": failure_reason if fallback_rows else "",
        "fallback_stage": failure_stage if fallback_rows else "",
        "fallback_rows": int(fallback_rows),
        "fallback_fraction": float(fallback_rows / n),
        "code_llm_mode": "code_llm_v2_scaffolded",
        "code_llm_scaffold_version": CONFIG.get("code_llm_scaffold_version", "Code-LLM-v2"),
        "code_llm_generation_success": bool(generation_success),
        "code_llm_compile_success": bool(compile_success),
        "code_llm_runtime_success": bool(runtime_success),
        "code_llm_validation_success": bool(validation_success),
        "code_llm_repair_success": bool(repair_success),
        "code_llm_fallback_used": bool(fallback_rows),
        "code_llm_fallback_reason": failure_reason if fallback_rows else "",
        "code_llm_failure_stage": failure_stage if fallback_rows else "",
        "llm_generated_rows": int(len(raw_df)),
        "llm_raw_rows": int(len(raw_df)),
        "llm_valid_rows": int(llm_valid_rows),
        "llm_valid_fraction": float(llm_valid_rows / len(raw_df)) if len(raw_df) else 0.0,
        "llm_calls": int(llm_calls),
        "llm_attempts": int(llm_calls),
        "llm_repair_attempts": int(max(0, llm_calls - 1)),
        "repair_attempts": int(max(0, llm_calls - 1)),
        "code_path": project_relative_path(code_path),
        "ollama_model": CONFIG["ollama_model"],
        "ollama_temperature": CONFIG["code_llm_temperature"],
        "ollama_top_p": CONFIG["ollama_top_p"],
        "ollama_num_predict": CONFIG["code_llm_num_predict"],
        "provenance_primary_source": "code_llm_sampler",
        "provenance_fallback_source": "code_llm_postprocess_fallback",
        "generation_runtime_seconds": float(time.time() - started),
    }
def generate_by_method(method, train_df, dataset, n, seed, code_path=None, train_size=None, audit_context=None):
    if method == "ctgan":
        return generate_ctgan(train_df, dataset, n, seed, epochs=CONFIG["ctgan_epochs"], method_label="ctgan")
    if method == "ctgan_strong":
        return generate_ctgan(train_df, dataset, n, seed, epochs=CONFIG["ctgan_strong_epochs"], method_label="ctgan_strong")
    if method == "direct_icl":
        return generate_direct_icl(train_df, dataset, n, seed, audit_context=audit_context, train_size=train_size)
    if method == "code_llm":
        return generate_code_llm(train_df, dataset, n, seed, code_path, audit_context=audit_context, train_size=train_size)
    if method == "fallback_only":
        return generate_fallback_only(train_df, dataset, n, seed)
    raise ValueError(method)


## Utility und Fidelity bewerten


In [9]:
def prepare_model_data(df, dataset):
    norm = normalize_to_schema(df, dataset)
    target = schema_for(dataset)["target"]
    X = norm.drop(columns=[target])
    y = target_to_binary(norm[target], dataset)
    return X, y
def build_classifier(dataset):
    schema = schema_for(dataset)
    target = schema["target"]
    numeric = [c for c in schema["numeric"] if c != target]
    categorical = [c for c in schema["categorical"] if c != target]
    transformers = []
    if numeric:
        transformers.append(("num", StandardScaler(), numeric))
    if categorical:
        transformers.append(("cat", make_one_hot_encoder(), categorical))
    return Pipeline([("preprocess", ColumnTransformer(transformers, remainder="drop")), ("classifier", LogisticRegression(max_iter=1000, random_state=0))])
def evaluate_majority_baseline(train_df, test_df, dataset):
    X_train, y_train = prepare_model_data(train_df, dataset)
    X_test, y_test = prepare_model_data(test_df, dataset)
    clf = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
    pred = clf.predict(X_test)
    return {"majority_baseline_f1": float(f1_score(y_test, pred, zero_division=0)), "majority_baseline_balanced_accuracy": float(balanced_accuracy_score(y_test, pred))}
def evaluate_real_train_upper_bound(train_df, test_df, dataset):
    X_train, y_train = prepare_model_data(train_df, dataset)
    X_test, y_test = prepare_model_data(test_df, dataset)
    if y_train.nunique() < 2 or y_test.nunique() < 2:
        return {"real_train_f1": np.nan, "real_train_balanced_accuracy": np.nan, "real_train_roc_auc": np.nan}
    clf = build_classifier(dataset).fit(X_train, y_train)
    pred = clf.predict(X_test)
    try:
        auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
    except Exception:
        auc = np.nan
    return {"real_train_f1": float(f1_score(y_test, pred, zero_division=0)), "real_train_balanced_accuracy": float(balanced_accuracy_score(y_test, pred)), "real_train_roc_auc": float(auc) if pd.notna(auc) else np.nan}
def utility_reference_deltas(f1, real):
    real_f1 = real.get("real_train_f1", np.nan)
    if pd.isna(f1) or pd.isna(real_f1):
        return {"tstr_f1_gap_to_real_train": np.nan, "tstr_f1_relative_to_real_train": np.nan}
    return {
        "tstr_f1_gap_to_real_train": float(real_f1 - f1),
        "tstr_f1_relative_to_real_train": float(f1 / real_f1) if real_f1 else np.nan,
    }

def evaluate_tstr_utility(synthetic_df, train_df, test_df, dataset):
    X_syn, y_syn = prepare_model_data(synthetic_df, dataset)
    X_test, y_test = prepare_model_data(test_df, dataset)
    base = evaluate_majority_baseline(train_df, test_df, dataset)
    real = evaluate_real_train_upper_bound(train_df, test_df, dataset)
    if y_syn.nunique() < 2 or y_test.nunique() < 2:
        reason = "single_class_synthetic_target" if y_syn.nunique() < 2 else "single_class_test_target"
        return {**base, **real, "utility_evaluable": False, "utility_failure_reason": reason, "tstr_f1": np.nan, "tstr_balanced_accuracy": np.nan, "tstr_roc_auc": np.nan, "tstr_f1_gain_over_majority": np.nan, **utility_reference_deltas(np.nan, real)}
    clf = build_classifier(dataset).fit(X_syn, y_syn)
    pred = clf.predict(X_test)
    try:
        auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
    except Exception:
        auc = np.nan
    f1 = float(f1_score(y_test, pred, zero_division=0))
    return {**base, **real, "utility_evaluable": True, "utility_failure_reason": "", "tstr_f1": f1, "tstr_balanced_accuracy": float(balanced_accuracy_score(y_test, pred)), "tstr_roc_auc": float(auc) if pd.notna(auc) else np.nan, "tstr_f1_gain_over_majority": float(f1 - base["majority_baseline_f1"]), **utility_reference_deltas(f1, real)}

def total_variation(real, syn, allowed=None):
    allowed = allowed or sorted(set(real.dropna()).union(set(syn.dropna())))
    p = real.value_counts(normalize=True).reindex(allowed, fill_value=0)
    q = syn.value_counts(normalize=True).reindex(allowed, fill_value=0)
    return float(0.5 * np.abs(p - q).sum())
def evaluate_target_conditioned_fidelity(real, syn, dataset):
    schema = schema_for(dataset)
    target = schema["target"]
    allowed_targets = schema["allowed_values"].get(target, sorted(set(real[target].dropna()).union(set(syn[target].dropna()))))
    num_dist, cat_dist, dependency_dist = [], [], []
    missing_class_count = 0
    for value in allowed_targets:
        real_sub = real[real[target] == value]
        syn_sub = syn[syn[target] == value]
        if len(real_sub) == 0 or len(syn_sub) == 0:
            missing_class_count += 1
            continue
        for col in schema["numeric"]:
            if col == target:
                continue
            try:
                num_dist.append(float(ks_2samp(pd.to_numeric(real_sub[col], errors="coerce").dropna(), pd.to_numeric(syn_sub[col], errors="coerce").dropna()).statistic))
            except Exception:
                pass
            low, high = schema["ranges"].get(col, (np.nan, np.nan))
            denom = float(high - low) if pd.notna(low) and pd.notna(high) and high != low else np.nan
            if denom and pd.notna(denom):
                rmean = pd.to_numeric(real_sub[col], errors="coerce").mean()
                smean = pd.to_numeric(syn_sub[col], errors="coerce").mean()
                if pd.notna(rmean) and pd.notna(smean):
                    dependency_dist.append(float(abs(rmean - smean) / denom))
        for col in schema["categorical"]:
            if col != target:
                cat_dist.append(total_variation(real_sub[col], syn_sub[col], schema["allowed_values"].get(col)))
    target_conditioned_numeric = float(np.mean(num_dist)) if num_dist else np.nan
    target_conditioned_categorical = float(np.mean(cat_dist)) if cat_dist else np.nan
    return {
        "target_conditioned_numeric_distance": target_conditioned_numeric,
        "target_conditioned_categorical_distance": target_conditioned_categorical,
        "target_conditioned_distribution_distance": float(np.nanmean([target_conditioned_numeric, target_conditioned_categorical])),
        "target_conditioned_missing_class_count": int(missing_class_count),
        "feature_target_mean_dependency_distance": float(np.mean(dependency_dist)) if dependency_dist else np.nan,
    }

def evaluate_fidelity(synthetic_df, train_df, dataset):
    schema = schema_for(dataset)
    syn = normalize_to_schema(synthetic_df, dataset)
    real = normalize_to_schema(train_df, dataset)
    target = schema["target"]
    num_dist, cat_dist = [], []
    for col in schema["numeric"]:
        if col != target:
            try:
                num_dist.append(float(ks_2samp(pd.to_numeric(real[col], errors="coerce").dropna(), pd.to_numeric(syn[col], errors="coerce").dropna()).statistic))
            except Exception:
                pass
    for col in schema["categorical"]:
        if col != target:
            cat_dist.append(total_variation(real[col], syn[col], schema["allowed_values"].get(col)))
    target_dist = total_variation(real[target], syn[target], schema["allowed_values"].get(target))
    corr_cols = [c for c in schema["numeric"] if c != target]
    corr, corr_reason = np.nan, ""
    if len(corr_cols) >= 2:
        corr = float(np.abs(real[corr_cols].astype(float).corr().fillna(0) - syn[corr_cols].astype(float).corr().fillna(0)).to_numpy().mean())
    else:
        corr_reason = "fewer_than_two_numeric_columns"
    numeric = float(np.mean(num_dist)) if num_dist else np.nan
    categorical = float(np.mean(cat_dist)) if cat_dist else np.nan
    overall = float(np.nanmean([numeric, categorical, target_dist, corr]))
    return {"numeric_fidelity_distance": numeric, "categorical_fidelity_distance": categorical, "target_distribution_distance": float(target_dist), "target_distribution_tv": float(target_dist), "correlation_distance": corr, "correlation_distance_reason": corr_reason, "distribution_distance_mean": overall, "ks_mean": numeric, "tv_mean": categorical, **evaluate_target_conditioned_fidelity(real, syn, dataset)}

def evaluate_privacy_and_diversity(synthetic_df, train_df, dataset):
    validation = validate_synthetic_data(synthetic_df, train_df, dataset)
    schema = schema_for(dataset)
    syn = normalize_to_schema(synthetic_df, dataset)
    real = normalize_to_schema(train_df, dataset)
    target = schema["target"]
    cat_div = []
    for col in schema["categorical"]:
        if col != target:
            cat_div.append(min(syn[col].nunique() / max(real[col].nunique(), 1), 1.0))
    std_ratios = []
    for col in schema["numeric"]:
        if col != target:
            rstd = pd.to_numeric(real[col], errors="coerce").std(ddof=0)
            sstd = pd.to_numeric(syn[col], errors="coerce").std(ddof=0)
            if rstd and rstd > 0:
                std_ratios.append(float(min(sstd / rstd, rstd / sstd) if sstd and sstd > 0 else 0.0))
    return {"duplicate_rate": validation["duplicate_rate"], "exact_train_duplicate_rate": validation["exact_train_duplicate_rate"], "unique_row_fraction": validation["unique_row_fraction"], "categorical_diversity_mean": float(np.mean(cat_div)) if cat_div else np.nan, "numeric_std_ratio": float(np.mean(std_ratios)) if std_ratios else np.nan}
def adult_demographic_parity(df):
    syn = normalize_to_schema(df, "adult")
    male = syn[syn["sex"] == "Male"]
    female = syn[syn["sex"] == "Female"]
    if len(male) == 0 or len(female) == 0:
        return {"evaluable": False, "male_rate": np.nan, "female_rate": np.nan, "diff": np.nan, "abs_diff": np.nan}
    male_rate = float((male["income"] == ">50K").mean())
    female_rate = float((female["income"] == ">50K").mean())
    diff = male_rate - female_rate
    return {"evaluable": True, "male_rate": male_rate, "female_rate": female_rate, "diff": float(diff), "abs_diff": float(abs(diff))}

def evaluate_fairness(synthetic_df, dataset, train_df=None, test_df=None):
    if dataset != "adult":
        return {"fairness_evaluable": False, "fairness_failure_reason": "no_protected_attribute", "positive_rate_male": np.nan, "positive_rate_female": np.nan, "adult_demographic_parity_difference": np.nan, "demographic_parity_difference_abs": np.nan, "real_train_demographic_parity_difference_abs": np.nan, "real_test_demographic_parity_difference_abs": np.nan, "dpd_abs_difference_to_real_train": np.nan, "dpd_abs_difference_to_real_test": np.nan}
    syn = adult_demographic_parity(synthetic_df)
    if not syn["evaluable"]:
        return {"fairness_evaluable": False, "fairness_failure_reason": "missing_protected_group", "positive_rate_male": np.nan, "positive_rate_female": np.nan, "adult_demographic_parity_difference": np.nan, "demographic_parity_difference_abs": np.nan, "real_train_demographic_parity_difference_abs": np.nan, "real_test_demographic_parity_difference_abs": np.nan, "dpd_abs_difference_to_real_train": np.nan, "dpd_abs_difference_to_real_test": np.nan}
    real_train = adult_demographic_parity(train_df) if train_df is not None else {"abs_diff": np.nan, "diff": np.nan}
    real_test = adult_demographic_parity(test_df) if test_df is not None else {"abs_diff": np.nan, "diff": np.nan}
    return {
        "fairness_evaluable": True,
        "fairness_failure_reason": "",
        "positive_rate_male": syn["male_rate"],
        "positive_rate_female": syn["female_rate"],
        "adult_demographic_parity_difference": syn["diff"],
        "demographic_parity_difference_abs": syn["abs_diff"],
        "real_train_adult_demographic_parity_difference": real_train.get("diff", np.nan),
        "real_train_demographic_parity_difference_abs": real_train.get("abs_diff", np.nan),
        "real_test_adult_demographic_parity_difference": real_test.get("diff", np.nan),
        "real_test_demographic_parity_difference_abs": real_test.get("abs_diff", np.nan),
        "dpd_abs_difference_to_real_train": float(abs(syn["abs_diff"] - real_train["abs_diff"])) if pd.notna(real_train.get("abs_diff", np.nan)) else np.nan,
        "dpd_abs_difference_to_real_test": float(abs(syn["abs_diff"] - real_test["abs_diff"])) if pd.notna(real_test.get("abs_diff", np.nan)) else np.nan,
    }

def evaluate_generated_dataframe(synthetic_df, train_df, test_df, dataset):
    start = time.time()
    validation = validate_synthetic_data(synthetic_df, train_df, dataset)
    validation_runtime = time.time() - start
    start = time.time()
    metrics = {**evaluate_tstr_utility(synthetic_df, train_df, test_df, dataset), **evaluate_fidelity(synthetic_df, train_df, dataset), **evaluate_privacy_and_diversity(synthetic_df, train_df, dataset), **evaluate_fairness(synthetic_df, dataset, train_df=train_df, test_df=test_df)}
    return {**validation, **metrics, "validation_runtime_seconds": float(validation_runtime), "evaluation_runtime_seconds": float(time.time() - start)}


## Experimente ausführen und Runs speichern


In [10]:
def result_defaults():
    return {
        "method_success": False, "effective_generator": "", "fallback_used": False, "fallback_reason": "",
        "fallback_stage": "", "fallback_rows": 0, "fallback_fraction": 0.0, "fallback_or_refill_rows": 0,
        "primary_generator_rows": np.nan, "repair_rows": 0, "repair_fraction": 0.0, "repair_runtime_seconds": 0.0,
        "llm_calls": 0, "llm_attempts": 0, "llm_repair_attempts": 0, "ollama_model": "",
        "ollama_temperature": np.nan, "ollama_top_p": np.nan, "ollama_num_predict": np.nan,
        "code_llm_mode": "", "code_llm_scaffold_version": "", "code_llm_generation_success": np.nan,
        "code_llm_compile_success": np.nan, "code_llm_runtime_success": np.nan, "code_llm_validation_success": np.nan,
        "code_llm_repair_success": np.nan, "code_llm_fallback_used": np.nan, "code_llm_fallback_reason": "",
        "code_llm_failure_stage": "", "llm_generated_rows": np.nan, "llm_valid_rows": np.nan,
        "llm_raw_rows": np.nan, "llm_valid_fraction": np.nan, "primary_only_rows": np.nan,
        "primary_only_fraction_of_target": np.nan, "primary_only_tstr_f1": np.nan,
        "primary_only_distribution_distance_mean": np.nan, "parse_error_count": np.nan,
        "invalid_format_rows": np.nan, "ctgan_epochs_used": np.nan, "ctgan_raw_sampled_rows": np.nan,
        "ctgan_raw_valid_rows": np.nan, "ctgan_raw_valid_rate": np.nan, "ctgan_rejection_rows": np.nan,
        "ctgan_repair_rows": np.nan, "ctgan_repair_fraction": np.nan, "fallback_only_rows": np.nan,
        "postprocessing_used": np.nan, "postprocessing_type": "", "raw_valid_rate": np.nan,
        "final_valid_rate": np.nan, "code_path": "", "timeout_used": False,
        "provenance_primary_source": "", "provenance_fallback_source": "",
    }

def prefix_metrics(metrics, prefix):
    return {f"{prefix}{key}": value for key, value in metrics.items()}

def run_single_experiment(dataset, train_size, seed, method, n=None, audit_context=None):
    n = int(n or CONFIG["synthetic_rows"])
    started = time.time()
    train_df, test_df = load_train_test_split(dataset, train_size, seed)
    code_path = GENERATED_CODE_DIR / f"code_llm_{dataset}_{train_size}_seed{seed}.py"
    base = {"dataset": dataset, "method": method, "train_size": int(train_size), "seed": int(seed), "target_rows": int(n), "status": "failed", "error": "", **result_defaults()}
    try:
        synthetic_df, generation = generate_by_method(method, train_df, dataset, n, seed, code_path if method == "code_llm" else None, train_size=train_size, audit_context=audit_context)
        diagnostic_frames = generation.pop("_diagnostic_frames", {})
        evaluation = evaluate_generated_dataframe(synthetic_df, train_df, test_df, dataset)
        result = {**base, **generation, **evaluation, "generated_rows": int(len(synthetic_df)), "status": "success" if len(synthetic_df) == n else "failed", "runtime_seconds": float(time.time() - started)}
        if "primary_valid_only" in diagnostic_frames:
            primary_df = diagnostic_frames["primary_valid_only"]
            result["primary_only_rows"] = int(len(primary_df))
            result["primary_only_fraction_of_target"] = float(len(primary_df) / n) if n else np.nan
            result.update(prefix_metrics(evaluate_generated_dataframe(primary_df, train_df, test_df, dataset), "primary_only_"))
        result["total_runtime_seconds"] = result["runtime_seconds"]
        result["repair_fraction"] = float(result.get("repair_fraction", result.get("repair_fallback_fraction", 0.0)))
        result["raw_generated_rows"] = int(result.get("raw_row_count", result.get("raw_generated_rows", result.get("llm_raw_rows", 0))) or 0)
        result["raw_valid_rows"] = int(result.get("raw_valid_row_count", result.get("ctgan_raw_valid_rows", result.get("llm_valid_rows", result.get("raw_valid_rows", 0)))) or 0)
        result["final_valid_rows"] = int(result.get("final_valid_row_count", len(synthetic_df)))
        result["fallback_rows"] = int(result.get("fallback_rows", result.get("fallback_or_refill_rows", 0)) or 0)
        result["fallback_or_refill_rows"] = int(result.get("fallback_or_refill_rows", result["fallback_rows"]) or 0)
        result["primary_generator_rows"] = int(result.get("primary_generator_rows", max(0, len(synthetic_df) - result["fallback_rows"])) or 0)
        if audit_context is not None:
            artifact_paths = {}
            if result["status"] == "success":
                final_csv = save_final_synthetic_csv(audit_context, synthetic_df, dataset, train_size, seed, method)
                provenance = save_row_provenance_sidecar(audit_context, result, dataset, train_size, seed, method, len(synthetic_df))
                if final_csv is not None:
                    artifact_paths["synthetic_final"] = project_relative_path(final_csv)
                if provenance is not None:
                    artifact_paths["row_provenance"] = project_relative_path(provenance)
            save_run_result_json(audit_context, result, dataset, train_size, seed, method, artifact_paths)
        return result
    except Exception as exc:
        failure_result = {**base, "status": "failed", "error": f"{type(exc).__name__}: {exc}", "runtime_seconds": float(time.time() - started), "total_runtime_seconds": float(time.time() - started)}
        if audit_context is not None:
            save_run_result_json(audit_context, failure_result, dataset, train_size, seed, method)
        return failure_result
def experiment_grid(smoke=False):
    datasets = CONFIG["datasets"][:1] if smoke else CONFIG["datasets"]
    train_sizes = CONFIG["train_sizes"][:1] if smoke else CONFIG["train_sizes"]
    seeds = CONFIG["seeds"][:1] if smoke else CONFIG["seeds"]
    for dataset in datasets:
        for train_size in train_sizes:
            for seed in seeds:
                for method in CONFIG["methods"]:
                    yield dataset, train_size, seed, method
def clean_generated_outputs():
    for directory in [TABLE_DIR, PLOT_DIR, GENERATED_CODE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
        for path in directory.glob("*"):
            if path.is_file():
                path.unlink()
def run_experiments(smoke=False, audit_context=None):
    n = CONFIG["smoke_synthetic_rows"] if smoke else CONFIG["synthetic_rows"]
    rows = []
    for dataset, train_size, seed, method in experiment_grid(smoke):
        result = run_single_experiment(dataset, train_size, seed, method, n, audit_context=None if smoke else audit_context)
        rows.append(result)
        if not smoke and CONFIG.get("save_incremental_results", True):
            append_incremental_result_csv(audit_context, result)
    return pd.DataFrame(rows)
def run_complete_project(rebuild_processed=False, rebuild_splits=False, run_smoke=True, run_full=True, clean_outputs=True, audit=True, run_id=None):
    if rebuild_processed:
        raise NotImplementedError("Processed datasets are expected as reproducible input files.")
    create_low_data_splits(rebuild=rebuild_splits)
    result = {"preflight": preflight_environment(timeout_seconds=5)}
    if run_smoke:
        if clean_outputs:
            clean_generated_outputs()
        result["smoke_results"] = run_experiments(smoke=True, audit_context=None)
        if clean_outputs and not run_full:
            clean_generated_outputs()
    if run_full:
        if clean_outputs:
            clean_generated_outputs()
        audit_context = create_audit_run_context(run_id) if (audit and CONFIG.get("audit_artifacts_enabled", True)) else None
        if audit_context is not None:
            result["audit_run_id"] = audit_context["run_id"]
            result["audit_run_dir"] = project_relative_path(audit_context["run_dir"])
        full = run_experiments(smoke=False, audit_context=audit_context)
        write_results_and_plots(full)
        snapshot_result_tables_and_plots(audit_context)
        if audit_context is not None:
            refresh_artifact_manifest(audit_context)
        result["full_results"] = full
        result["validation"] = validate_submission_outputs()
    return result


## Ergebnistabellen erstellen


In [11]:
MAIN_COLUMNS = [
    "dataset", "method", "train_size", "seed", "status", "method_success", "effective_generator",
    "tstr_f1", "tstr_balanced_accuracy", "tstr_roc_auc", "real_train_f1", "majority_baseline_f1",
    "tstr_f1_gap_to_real_train", "tstr_f1_relative_to_real_train",
    "distribution_distance_mean", "numeric_fidelity_distance", "categorical_fidelity_distance",
    "target_distribution_distance", "target_conditioned_distribution_distance", "feature_target_mean_dependency_distance",
    "constraint_violation_rate", "duplicate_rate", "exact_train_duplicate_rate",
    "primary_generator_rows", "fallback_rows", "fallback_fraction", "repair_fraction",
    "primary_only_rows", "primary_only_tstr_f1", "primary_only_distribution_distance_mean",
    "ctgan_epochs_used", "runtime_seconds",
]
SUMMARY_METRICS = [
    "tstr_f1", "tstr_balanced_accuracy", "tstr_roc_auc", "tstr_f1_gap_to_real_train",
    "tstr_f1_relative_to_real_train", "distribution_distance_mean", "numeric_fidelity_distance",
    "categorical_fidelity_distance", "target_distribution_distance", "target_conditioned_distribution_distance",
    "feature_target_mean_dependency_distance", "constraint_violation_rate", "duplicate_rate",
    "exact_train_duplicate_rate", "runtime_seconds", "fallback_fraction", "repair_fraction",
    "raw_valid_rate", "primary_only_tstr_f1", "primary_only_distribution_distance_mean",
    "demographic_parity_difference_abs", "real_train_demographic_parity_difference_abs",
    "dpd_abs_difference_to_real_train",
]
def build_main_results(full_df):
    return full_df[[c for c in MAIN_COLUMNS if c in full_df.columns]].copy()
def numeric_series(group, column):
    if column not in group.columns:
        return pd.Series([np.nan] * len(group), index=group.index, dtype="float64")
    return pd.to_numeric(group[column], errors="coerce")
def bool_mean(group, column):
    if column not in group.columns:
        return np.nan
    return float(group[column].fillna(False).astype(bool).mean())
def aggregate_summary(full_df):
    rows = []
    for keys, group in full_df.groupby(["dataset", "method", "train_size"], dropna=False):
        row = {
            "dataset": keys[0],
            "method": keys[1],
            "train_size": keys[2],
            "n_runs": int(len(group)),
            "n_success": int((group["status"] == "success").sum()) if "status" in group.columns else 0,
            "success_rate": float((group["status"] == "success").mean()) if "status" in group.columns else np.nan,
            "method_success_rate": bool_mean(group, "method_success"),
            "fallback_rate": bool_mean(group, "fallback_used"),
            "mean_fallback_fraction": float(numeric_series(group, "fallback_fraction").mean()),
            "mean_repair_fraction": float(numeric_series(group, "repair_fraction").mean()),
            "mean_raw_valid_rate": float(numeric_series(group, "raw_valid_rate").mean()),
            "mean_runtime_seconds": float(numeric_series(group, "runtime_seconds").mean()),
        }
        for metric in SUMMARY_METRICS:
            if metric in group.columns:
                values = numeric_series(group, metric)
                row[f"{metric}_mean"] = float(values.mean()) if values.notna().any() else np.nan
                row[f"{metric}_std"] = float(values.std(ddof=1)) if values.notna().sum() > 1 else np.nan
                row[f"{metric}_min"] = float(values.min()) if values.notna().any() else np.nan
                row[f"{metric}_max"] = float(values.max()) if values.notna().any() else np.nan
                row[f"{metric}_n"] = int(values.notna().sum())
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["dataset", "train_size", "method"]).reset_index(drop=True)
def aggregate_provenance_ablation_summary(full_df):
    rows = []
    for keys, group in full_df.groupby(["dataset", "method", "train_size"], dropna=False):
        fallback_rows = numeric_series(group, "fallback_rows")
        primary_rows = numeric_series(group, "primary_generator_rows")
        row = {
            "dataset": keys[0],
            "method": keys[1],
            "train_size": keys[2],
            "n_runs": int(len(group)),
            "fallback_runs": int(fallback_rows.fillna(0).gt(0).sum()),
            "fallback_rows_total": int(fallback_rows.fillna(0).sum()),
            "primary_generator_rows_mean": float(primary_rows.mean()) if primary_rows.notna().any() else np.nan,
            "fallback_fraction_mean": float(numeric_series(group, "fallback_fraction").mean()),
            "raw_valid_rate_mean": float(numeric_series(group, "raw_valid_rate").mean()),
            "final_tstr_f1_mean": float(numeric_series(group, "tstr_f1").mean()),
            "final_fidelity_distance_mean": float(numeric_series(group, "distribution_distance_mean").mean()),
            "primary_only_rows_mean": float(numeric_series(group, "primary_only_rows").mean()),
            "primary_only_tstr_f1_mean": float(numeric_series(group, "primary_only_tstr_f1").mean()),
            "primary_only_fidelity_distance_mean": float(numeric_series(group, "primary_only_distribution_distance_mean").mean()),
            "tstr_f1_gap_to_real_train_mean": float(numeric_series(group, "tstr_f1_gap_to_real_train").mean()),
        }
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["dataset", "train_size", "method"]).reset_index(drop=True)
def write_result_tables(full_df):
    full_df.to_csv(TABLE_DIR / "full_experiment_results.csv", index=False)
    build_main_results(full_df).to_csv(TABLE_DIR / "main_results.csv", index=False)
    aggregate_summary(full_df).to_csv(TABLE_DIR / "summary_by_method_dataset_train_size.csv", index=False)
    aggregate_provenance_ablation_summary(full_df).to_csv(TABLE_DIR / "provenance_ablation_summary.csv", index=False)


## Plots erzeugen


In [12]:
def plot_grouped_metric(full_df, metric, output_path, title, ylabel, adult_only=False, log_y=False):
    df = full_df.copy()
    if adult_only:
        df = df[df["dataset"] == "adult"]
    if metric not in df.columns:
        df[metric] = np.nan
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    if df[metric].notna().sum() == 0:
        ax.text(0.5, 0.5, f"Keine auswertbaren Werte für {metric}", ha="center", va="center")
        ax.set_axis_off()
    else:
        groups = df.groupby(["dataset", "train_size", "method"], dropna=False)[metric].agg(["mean", "std"]).reset_index()
        combo_order = groups[["dataset", "train_size"]].drop_duplicates().sort_values(["dataset", "train_size"]).reset_index(drop=True)
        method_order = [m for m in CONFIG["methods"] if m in set(groups["method"])]
        x = np.arange(len(combo_order))
        width = min(0.8 / max(len(method_order), 1), 0.18)
        all_means = []
        for idx, method in enumerate(method_order):
            means, stds = [], []
            for _, combo in combo_order.iterrows():
                row = groups[(groups["dataset"] == combo["dataset"]) & (groups["train_size"] == combo["train_size"]) & (groups["method"] == method)]
                means.append(float(row["mean"].iloc[0]) if len(row) else np.nan)
                stds.append(float(row["std"].fillna(0).iloc[0]) if len(row) else 0.0)
            all_means.extend(means)
            offset = (idx - (len(method_order) - 1) / 2) * width
            ax.bar(x + offset, means, width=width, yerr=stds, capsize=3, label=method.replace("_", "-"))
        labels = [f"{str(r.dataset).capitalize()} n={int(r.train_size)}" for r in combo_order.itertuples()]
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=0, ha="center")
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.legend(loc="best", fontsize="small")
        ax.grid(axis="y", alpha=0.25)
        if log_y and pd.Series(all_means).gt(0).any():
            ax.set_yscale("log")
        if metric == "constraint_violation_rate" and float(groups["mean"].fillna(0).max()) == 0.0:
            ax.set_ylim(0, 0.05)
            ax.text(0.5, 0.9, "Alle finalen Outputs bestehen die Constraint-Prüfung", transform=ax.transAxes, ha="center")
    fig.tight_layout()
    fig.savefig(output_path, dpi=220)
    plt.close(fig)
def create_plots(full_df):
    plot_grouped_metric(full_df, "tstr_f1", PLOT_DIR / "tstr_f1.png", "TSTR-F1", "F1-Wert")
    plot_grouped_metric(full_df, "distribution_distance_mean", PLOT_DIR / "fidelity_distance.png", "Zusammengesetzte Fidelity-Distanz", "Distanz")
    plot_grouped_metric(full_df, "constraint_violation_rate", PLOT_DIR / "constraint_violation_rate.png", "Constraint-Verletzungsrate", "Verletzungsrate")
    plot_grouped_metric(full_df, "runtime_seconds", PLOT_DIR / "runtime_seconds.png", "Laufzeit", "Sekunden", log_y=True)
    metric = "demographic_parity_difference_abs" if "demographic_parity_difference_abs" in full_df.columns else "adult_demographic_parity_difference"
    plot_grouped_metric(full_df, metric, PLOT_DIR / "adult_demographic_parity_difference.png", "Adult-Demographic-Parity-Differenz", "Absolute Differenz", adult_only=True)
def write_results_and_plots(full_df):
    write_result_tables(full_df)
    create_plots(full_df)


## Validierung, Audit und Abgabechecks


In [13]:
REQUIRED_TABLES = [
    TABLE_DIR / "full_experiment_results.csv",
    TABLE_DIR / "main_results.csv",
    TABLE_DIR / "summary_by_method_dataset_train_size.csv",
    TABLE_DIR / "provenance_ablation_summary.csv",
]
REQUIRED_PLOTS = [
    PLOT_DIR / "tstr_f1.png",
    PLOT_DIR / "fidelity_distance.png",
    PLOT_DIR / "constraint_violation_rate.png",
    PLOT_DIR / "runtime_seconds.png",
    PLOT_DIR / "adult_demographic_parity_difference.png",
]
REQUIRED_RESULT_COLUMNS = [
    "method_success", "effective_generator", "fallback_fraction", "repair_fraction",
    "raw_valid_rate", "utility_evaluable", "tstr_f1", "distribution_distance_mean",
    "exact_train_duplicate_rate", "tstr_f1_gap_to_real_train",
    "target_conditioned_distribution_distance", "primary_generator_rows",
]
AUDIT_RUN_TOP_LEVEL_FILES = {"pip_freeze.txt", "environment_snapshot.json", "config_snapshot.json", "data_hashes.json", "artifact_manifest.json"}
AUDIT_RUN_SUBDIRS = {"tables", "plots", "synthetic_final", "raw_llm", "code_llm_traces", "row_provenance", "run_results"}
ABSOLUTE_LOCAL_PATH_RE = re.compile(r"(/Users/|/home/|/mnt/data/|[A-Za-z]:\\\\)")

def preflight_environment(timeout_seconds=5):
    result = {"python_ready": True, "inputs_ready": False, "sdv_available": CTGANSynthesizer is not None, "ollama_available": False, "ollama_model_available": False, "ready_for_smoke_test": False, "ready_for_experiments": False, "messages": []}
    try:
        check_required_inputs()
        result["inputs_ready"] = True
    except Exception as exc:
        result["messages"].append(str(exc))
    try:
        response = requests.get(CONFIG["ollama_endpoint"].rstrip("/") + "/api/tags", timeout=timeout_seconds)
        if response.ok:
            result["ollama_available"] = True
            names = [m.get("name", "") for m in response.json().get("models", [])]
            result["ollama_model_available"] = CONFIG["ollama_model"] in names or any(name.startswith(CONFIG["ollama_model"]) for name in names)
    except Exception as exc:
        result["messages"].append(f"Ollama not reachable: {exc}")
    result["ready_for_smoke_test"] = bool(result["inputs_ready"] and result["sdv_available"] and result["ollama_available"] and result["ollama_model_available"])
    result["ready_for_experiments"] = result["ready_for_smoke_test"]
    return result

def validate_static_configuration():
    assert planned_experiment_count() == 60
    assert CONFIG["methods"] == ["ctgan", "ctgan_strong", "direct_icl", "code_llm", "fallback_only"]
    assert CONFIG["datasets"] == ["adult", "pima"]
    assert CONFIG["train_sizes"] == [100, 500]
    assert CONFIG["seeds"] == [42, 123, 2025]
    assert CONFIG["ollama_model"] == "llama3.2:3b"
    assert CONFIG["direct_icl_temperature"] == 0.5
    assert CONFIG["code_llm_temperature"] == 0.1
    assert CONFIG["ollama_top_p"] == 0.9
    assert CONFIG["direct_icl_num_predict"] == 2200
    assert CONFIG["code_llm_num_predict"] == 1800
    assert CONFIG["ctgan_epochs"] == 50
    assert CONFIG["ctgan_strong_epochs"] == 300
    assert CONFIG["method_success_min_llm_fraction"] == 0.5
    assert CONFIG.get("code_llm_scaffold_version") == "Code-LLM-v2"
    return {"ok": True, "planned_experiments": planned_experiment_count()}

def expected_combinations():
    return {(d, ts, s, m) for d in CONFIG["datasets"] for ts in CONFIG["train_sizes"] for s in CONFIG["seeds"] for m in CONFIG["methods"]}

def forbidden_zip_member(name):
    normalized = name.replace("\\", "/")
    parts = normalized.split("/")
    forbidden_exact = {".DS_Store", ".python-version"}
    forbidden_parts = {"__MACOSX", ".venv", "venv", "env", ".ipynb_checkpoints", "__pycache__", ".pytest_cache", ".mypy_cache", ".ruff_cache", ".cache", ".ollama"}
    forbidden_suffixes = (".pyc", ".pyo", ".so", ".dylib")
    forbidden_substrings = ["/data/raw/", "/data/synthetic/", "/ollama/models/", "/models/blobs/", "/models/manifests/", "smoke_test_results.csv", "processed_data_summary.json", "adult_split_summary.json", "pima_split_summary.json"]
    if any(part in forbidden_exact or part in forbidden_parts or part.startswith("._") for part in parts):
        return True
    if normalized.endswith(forbidden_suffixes):
        return True
    padded = "/" + normalized
    return any(token in padded for token in forbidden_substrings)

def is_allowed_audit_artifact(path):
    path = Path(path)
    try:
        rel_parts = path.relative_to(RUNS_DIR).parts
    except Exception:
        return False
    if len(rel_parts) < 2:
        return False
    run_id, *tail = rel_parts
    if len(tail) == 1 and tail[0] in AUDIT_RUN_TOP_LEVEL_FILES:
        return True
    if tail[0] not in AUDIT_RUN_SUBDIRS:
        return False
    return path.suffix.lower() in {".json", ".csv", ".png", ".txt"}

def validate_notebook_json(notebook_path):
    try:
        payload = json.loads(Path(notebook_path).read_text(encoding="utf-8"))
        if payload.get("nbformat") != 4 or "cells" not in payload:
            return [f"Notebook is not a valid nbformat-4 JSON file: {notebook_path}"]
        return []
    except Exception as exc:
        return [f"Notebook JSON invalid: {notebook_path}: {type(exc).__name__}: {exc}"]

def parse_synthetic_filename(path):
    stem = Path(path).stem
    for method in sorted(CONFIG["methods"], key=len, reverse=True):
        suffix = f"_{method}"
        if stem.endswith(suffix):
            prefix = stem[:-len(suffix)]
            parts = prefix.split("_")
            if len(parts) == 3 and parts[2].startswith("seed"):
                return parts[0], int(parts[1]), int(parts[2].replace("seed", "")), method
    raise ValueError(f"Cannot parse synthetic filename: {path.name}")


def validate_future_synthetic_csvs():
    errors = []
    provenance_columns = {"audit_id", "provenance", "source", "fallback_source", "row_provenance"}
    validity_flags = [
        "format_valid", "schema_valid", "category_valid", "range_valid", "constraint_valid",
        "target_valid", "duplicate_valid", "privacy_valid",
    ]
    for path in sorted(RUNS_DIR.glob("*/synthetic_final/*.csv")):
        try:
            dataset, train_size, seed, method = parse_synthetic_filename(path)
            df = pd.read_csv(path)
            expected_columns = schema_for(dataset)["columns"]
            if list(df.columns) != expected_columns:
                errors.append(f"Synthetic CSV schema mismatch: {project_relative_path(path)}")
            if len(df) != CONFIG["synthetic_rows"]:
                errors.append(f"Synthetic CSV row count mismatch: {project_relative_path(path)} has {len(df)} rows")
            extras = sorted(provenance_columns.intersection(df.columns))
            if extras:
                errors.append(f"Synthetic CSV contains provenance/audit columns {extras}: {project_relative_path(path)}")
            train_df, _ = load_train_test_split(dataset, train_size, seed)
            checks = validate_synthetic_data(df, train_df, dataset)
            for flag in validity_flags:
                if not bool(checks.get(flag, False)):
                    errors.append(f"Synthetic CSV failed {flag}: {project_relative_path(path)}")
            if int(checks.get("raw_valid_row_count", -1)) != len(df):
                errors.append(f"Synthetic CSV has invalid rows after schema checks: {project_relative_path(path)}")
            if float(checks.get("duplicate_rate", 1.0)) != 0.0:
                errors.append(f"Synthetic CSV has internal duplicates: {project_relative_path(path)}")
            if float(checks.get("exact_train_duplicate_rate", 1.0)) != 0.0:
                errors.append(f"Synthetic CSV has exact train duplicates: {project_relative_path(path)}")
        except Exception as exc:
            errors.append(f"Synthetic CSV validation failed: {project_relative_path(path)}: {type(exc).__name__}: {exc}")
    return errors
def validate_future_provenance_files():
    errors = []
    required = {"dataset", "train_size", "seed", "method", "final_rows", "primary_generator_rows", "fallback_or_refill_rows", "fallback_fraction", "method_success", "method_success_threshold", "provenance_note"}
    for path in sorted(RUNS_DIR.glob("*/row_provenance/*_provenance.json")):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            missing = required - set(payload)
            if missing:
                errors.append(f"Provenance missing keys {sorted(missing)}: {project_relative_path(path)}")
                continue
            final_rows = int(payload["final_rows"])
            primary_rows = int(payload["primary_generator_rows"])
            fallback_rows = int(payload["fallback_or_refill_rows"])
            fraction = float(payload["fallback_fraction"])
            if primary_rows + fallback_rows != final_rows:
                errors.append(f"Provenance row counts inconsistent: {project_relative_path(path)}")
            expected_fraction = fallback_rows / final_rows if final_rows else 0.0
            if abs(fraction - expected_fraction) > 1e-9:
                errors.append(f"Provenance fallback_fraction inconsistent: {project_relative_path(path)}")
            row_csv = payload.get("row_level_provenance_csv", "")
            if row_csv:
                row_csv_path = PROJECT_ROOT / row_csv
                if not row_csv_path.exists():
                    errors.append(f"Row-level provenance CSV missing: {row_csv}")
                else:
                    row_df = pd.read_csv(row_csv_path)
                    if len(row_df) != final_rows:
                        errors.append(f"Row-level provenance row count mismatch: {row_csv}")
                    if "is_refill" in row_df.columns and int(row_df["is_refill"].astype(bool).sum()) != fallback_rows:
                        errors.append(f"Row-level provenance refill count mismatch: {row_csv}")
        except Exception as exc:
            errors.append(f"Provenance JSON invalid: {project_relative_path(path)}: {type(exc).__name__}: {exc}")
    return errors


def validate_no_absolute_local_paths():
    errors = []
    excluded_names = {"environment_snapshot.json", "pip_freeze.txt"}
    text_suffixes = {".csv", ".json", ".txt", ".md"}
    for base in [TABLE_DIR, RUNS_DIR]:
        if not base.exists():
            continue
        for path in sorted(p for p in base.rglob("*") if p.is_file() and p.suffix.lower() in text_suffixes):
            if path.name in excluded_names:
                continue
            try:
                text = path.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                continue
            if ABSOLUTE_LOCAL_PATH_RE.search(text):
                errors.append(f"Absolute local path found in portable result artifact: {project_relative_path(path)}")
    return errors
def validate_audit_run_completeness(latest_only=True):
    errors = []
    if not RUNS_DIR.exists():
        return errors
    run_dirs = sorted(p for p in RUNS_DIR.iterdir() if p.is_dir() and not p.name.startswith("."))
    if latest_only and run_dirs:
        run_dirs = [run_dirs[-1]]
    expected_runs = planned_experiment_count()
    required_top_level = AUDIT_RUN_TOP_LEVEL_FILES
    required_subdirs = ["tables", "plots", "synthetic_final", "raw_llm/direct_icl", "code_llm_traces", "row_provenance", "run_results"]
    for run_dir in run_dirs:
        for name in required_top_level:
            if not (run_dir / name).exists():
                errors.append(f"Audit run missing {name}: {project_relative_path(run_dir)}")
        for name in required_subdirs:
            if not (run_dir / name).exists():
                errors.append(f"Audit run missing directory {name}: {project_relative_path(run_dir)}")
        synthetic = sorted((run_dir / "synthetic_final").glob("*.csv")) if (run_dir / "synthetic_final").exists() else []
        provenance = sorted((run_dir / "row_provenance").glob("*_provenance.json")) if (run_dir / "row_provenance").exists() else []
        result_json = sorted((run_dir / "run_results").glob("*_result.json")) if (run_dir / "run_results").exists() else []
        direct_logs = sorted((run_dir / "raw_llm" / "direct_icl").glob("*.json")) if (run_dir / "raw_llm" / "direct_icl").exists() else []
        code_traces = sorted((run_dir / "code_llm_traces").glob("*.json")) if (run_dir / "code_llm_traces").exists() else []
        if len(synthetic) != expected_runs:
            errors.append(f"Audit run expected {expected_runs} final synthetic CSVs, found {len(synthetic)}: {project_relative_path(run_dir)}")
        if len(provenance) != expected_runs:
            errors.append(f"Audit run expected {expected_runs} provenance JSONs, found {len(provenance)}: {project_relative_path(run_dir)}")
        if len(result_json) != expected_runs:
            errors.append(f"Audit run expected {expected_runs} per-run result JSONs, found {len(result_json)}: {project_relative_path(run_dir)}")
        if not (run_dir / "run_results" / "incremental_results.csv").exists():
            errors.append(f"Audit run missing incremental_results.csv: {project_relative_path(run_dir)}")
        if len(direct_logs) == 0:
            errors.append(f"Audit run missing Direct-ICL raw logs: {project_relative_path(run_dir)}")
        if len(code_traces) == 0:
            errors.append(f"Audit run missing Code-LLM traces: {project_relative_path(run_dir)}")
    return errors

def validate_submission_zip(zip_path):
    errors = []
    try:
        with zipfile.ZipFile(zip_path) as zf:
            bad = zf.testzip()
            if bad:
                errors.append(f"Unreadable ZIP member: {bad}")
            forbidden = [name for name in zf.namelist() if forbidden_zip_member(name)]
            if forbidden:
                errors.append(f"Forbidden ZIP members: {forbidden[:10]}")
            if not any(name.endswith("README.md") for name in zf.namelist()):
                errors.append("ZIP missing README.md")
            if not any(name.endswith("requirements.txt") for name in zf.namelist()):
                errors.append("ZIP missing requirements.txt")
    except Exception as exc:
        errors.append(f"ZIP not readable: {zip_path}: {type(exc).__name__}: {exc}")
    return errors

def validate_submission_outputs(zip_path=None):
    errors, warnings = [], []
    for path in [PROJECT_ROOT / "README.md", PROJECT_ROOT / "requirements.txt"]:
        if not path.exists():
            errors.append(f"Missing file: {path}")
    notebook_path = PROJECT_ROOT / "llm_tabular_synthesis_abgabe_final.ipynb"
    if not notebook_path.exists():
        errors.append(f"Missing notebook: {notebook_path}")
    else:
        errors.extend(validate_notebook_json(notebook_path))
    try:
        check_required_inputs()
    except Exception as exc:
        errors.append(str(exc))
    for path in REQUIRED_TABLES:
        if not path.exists():
            errors.append(f"Missing result table: {path}")
    for path in REQUIRED_PLOTS:
        if not path.exists() or path.stat().st_size == 0:
            errors.append(f"Missing or empty plot: {path}")
    if not GENERATED_CODE_DIR.exists():
        errors.append(f"Missing generated_code directory: {GENERATED_CODE_DIR}")
    code_files = sorted(GENERATED_CODE_DIR.glob("code_llm_*.py"))
    if len(code_files) != 12:
        errors.append(f"Expected 12 code_llm files, found {len(code_files)}")
    full_path = TABLE_DIR / "full_experiment_results.csv"
    if full_path.exists():
        full = pd.read_csv(full_path)
        if len(full) != planned_experiment_count():
            errors.append(f"Expected 36 result rows, found {len(full)}")
        combos = set(zip(full["dataset"], full["train_size"], full["seed"], full["method"]))
        missing = expected_combinations() - combos
        extra = combos - expected_combinations()
        if missing:
            errors.append(f"Missing experiment combinations: {sorted(missing)[:5]}")
        if extra:
            errors.append(f"Unexpected experiment combinations: {sorted(extra)[:5]}")
        for col in REQUIRED_RESULT_COLUMNS:
            if col not in full.columns:
                errors.append(f"Missing required result column: {col}")
        if "code_llm_fallback_used" in full:
            code_llm = full[full["method"] == "code_llm"]
            rate = code_llm["code_llm_fallback_used"].astype(str).str.lower().isin(["true", "1"]).mean() if len(code_llm) else 0
            if rate > 0.5:
                warnings.append(f"code_llm fallback rate > 0.5: {rate:.2f}")
    for path in sorted(RUNS_DIR.rglob("*")):
        if path.is_file() and not is_allowed_audit_artifact(path):
            errors.append(f"Unexpected audit artifact path: {project_relative_path(path)}")
    errors.extend(validate_audit_run_completeness())
    errors.extend(validate_future_synthetic_csvs())
    errors.extend(validate_future_provenance_files())
    errors.extend(validate_no_absolute_local_paths())
    candidate_zip = Path(zip_path) if zip_path is not None else PROJECT_ROOT.parent / "Grundprojekt_Code_final.zip"
    if zip_path is not None or candidate_zip.exists():
        errors.extend(validate_submission_zip(candidate_zip))
    return {"ok": len(errors) == 0, "errors": errors, "warnings": warnings}


## Artefakte sammeln und ZIP erstellen


In [14]:
def audit_artifact_files():
    if not RUNS_DIR.exists():
        return []
    return sorted(path for path in RUNS_DIR.rglob("*") if path.is_file() and is_allowed_audit_artifact(path))

def allowlisted_submission_files():
    files = [
        PROJECT_ROOT / "llm_tabular_synthesis_abgabe_final.ipynb",
        PROJECT_ROOT / "README.md",
        PROJECT_ROOT / "requirements.txt",
        PROCESSED_DIR / "adult_processed.csv",
        PROCESSED_DIR / "pima_processed.csv",
        SPLIT_DIR / "adult_test.csv",
        SPLIT_DIR / "pima_test.csv",
    ]
    for dataset in CONFIG["datasets"]:
        for train_size in CONFIG["train_sizes"]:
            for seed in CONFIG["seeds"]:
                files.append(SPLIT_DIR / f"{dataset}_train_{train_size}_seed{seed}.csv")
    files.extend(sorted(GENERATED_CODE_DIR.glob("code_llm_*.py")))
    files.extend(REQUIRED_TABLES)
    files.extend(REQUIRED_PLOTS)
    files.extend(audit_artifact_files())
    return files
def create_submission_zip(zip_path=None):
    ensure_project_support_files()
    zip_path = Path(zip_path) if zip_path is not None else PROJECT_ROOT.parent / "Grundprojekt_Code_final.zip"
    files = allowlisted_submission_files()
    missing = [project_relative_path(path) for path in files if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing files for submission ZIP:\n" + "\n".join(missing))
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in files:
            arcname = "Grundprojekt_Code/" + path.relative_to(PROJECT_ROOT).as_posix()
            if forbidden_zip_member(arcname):
                raise ValueError(f"Forbidden ZIP member: {arcname}")
            zf.write(path, arcname)
    with zipfile.ZipFile(zip_path) as zf:
        forbidden = [name for name in zf.namelist() if forbidden_zip_member(name)]
    if forbidden:
        raise ValueError(f"Forbidden ZIP members: {forbidden}")
    return zip_path

## Hauptlauf steuern


In [ ]:
def main(run_full=False, rebuild_processed=False, rebuild_splits=False, run_smoke=False, clean_outputs=False, audit=None):
    validate_static_configuration()
    if rebuild_processed:
        raise NotImplementedError("Processed datasets are expected as reproducible input files.")
    if CONFIG.get("auto_create_missing_splits", True):
        ensure_project_inputs(require_splits=False)
        create_low_data_splits(rebuild=rebuild_splits)
    else:
        check_required_inputs()
    ensure_project_support_files()
    preflight_result = preflight_environment(timeout_seconds=5)
    if not preflight_result.get("ready_for_experiments", False):
        raise RuntimeError(f"Preflight failed: {preflight_result}")
    project_result = run_complete_project(
        rebuild_processed=False,
        rebuild_splits=rebuild_splits,
        run_smoke=run_smoke,
        run_full=run_full,
        clean_outputs=clean_outputs,
        audit=CONFIG.get("audit_artifacts_enabled", True) if audit is None else audit,
    )
    zip_path = create_submission_zip() if run_full else None
    validation_result = validate_submission_outputs(zip_path) if run_full else {"ok": True, "skipped": "run_full=False"}
    summary = {
        "preflight": preflight_result,
        "project_result_keys": list(project_result.keys()),
        "validation_result": validation_result,
        "zip_path": project_relative_path(zip_path) if zip_path else "",
    }
    print(json.dumps(json_safe(summary), indent=2, ensure_ascii=False))
    return summary


if __name__ == "__main__":
    main()
